# MasterMind AI Full Project Notebook

This notebook is a **single-file notebook conversion** of the project source, grouped by module.

Important notes:
- The code is embedded directly into this notebook so it does not depend on your local `.py` files.
- It is grouped into the module structure you requested.
- Because the original project uses external packages such as Flask, pandas, scikit-learn, SHAP, and XGBoost, those package dependencies still exist at the Python-environment level.
- What has been removed is the dependency on your project file structure like `src/...` and `configs/...`.


## How To Use This Notebook

1. Read or run the support cells first.
2. Then move module by module from Module 1 to Module 5.
3. The code in each section is copied from your project files so the notebook acts like a code archive in module form.
4. Some module code still imports external libraries, so those libraries must be available in the Python environment if you want to execute every cell.


In [ ]:
# This bootstrap cell is optional.
# It simply records where this notebook was generated from and helps readers understand context.
from pathlib import Path
NOTEBOOK_PATH = Path.cwd()
print('Notebook working directory:', NOTEBOOK_PATH)


## Support: Project Bootstrap\n
\n
This cell defines a project root helper so the notebook is easier to understand when opened in Jupyter.\n
\n
Original source: generated notebook bootstrap cell.\n

## Support: Shared Config\n
\n
These are the shared configuration constants used across the project. They are embedded directly here so the notebook does not depend on configs/config.py.\n
\n
Original source: `configs/config.py`\n

In [ ]:
"""
MasterMind — Shared Configuration
All project-wide constants. Every module imports from here.
"""

import os

# ─── Reproducibility ─────────────────────────────────────
RANDOM_STATE: int = 42

# ─── Data Split Fractions ────────────────────────────────
TRAIN_FRAC: float = 0.60
VAL_MODEL_FRAC: float = 0.10
VAL_POLICY_FRAC: float = 0.10
TEST_FRAC: float = 0.20

# ─── Feature Engineering ─────────────────────────────────
RARE_CATEGORY_MIN_COUNT: int = 500
MISSING_RATE_THRESHOLD: float = 0.05

# ─── Decision Policy ─────────────────────────────────────
APPROVE_THRESHOLD: float = 0.15
DECLINE_THRESHOLD: float = 0.35

# ─── Fairness Audit ──────────────────────────────────────
FAIRNESS_MIN_N: int = 200
FAIRNESS_MIN_DEFAULTS: int = 20

# ─── Paths ───────────────────────────────────────────────
DATA_DIR: str = os.environ.get(
    "DATA_PROCESSED_DIR", "data/processed/")
ARTIFACT_DIR: str = os.environ.get(
    "ARTIFACT_DIR", "artifacts/")

# ─── Versioning ──────────────────────────────────────────
FAIRNESS_AUDIT_VERSION: str = "proxy_audit_2026Q1_v1.0"
MODEL_VERSIONS: dict = {
    "full": "full_v2.1.0",
    "reduced": "reduced_v2.1.0",
}

# ─── API Section Requirements ────────────────────────────
FULL_REQUIRED_SECTIONS: set[str] = {
    "application", "bureau_agg", "previous_agg",
    "installments_agg", "pos_cash_agg", "credit_card_agg",
}


# ─── Self-verification ──────────────────────────────────
if __name__ == "__main__":
    import importlib
    cfg = importlib.import_module("configs.config")
    assert cfg.RANDOM_STATE == 42
    assert cfg.APPROVE_THRESHOLD == 0.15
    assert cfg.DECLINE_THRESHOLD == 0.35
    assert cfg.FAIRNESS_AUDIT_VERSION == "proxy_audit_2026Q1_v1.0"
    assert "full" in cfg.MODEL_VERSIONS
    print("configs/config.py verified ✓")


## Support: Runtime Verification\n
\n
These helper functions handle schema checks, fingerprints, and artifact validation. They are embedded so later modules can reference them without importing local project files.\n
\n
Original source: `src/runtime_verification.py`\n

In [ ]:
"""Shared verification and lineage helpers for runtime artifacts."""

from __future__ import annotations

import hashlib
import json
import os
import subprocess
from typing import Any, Iterable, Mapping

import numpy as np
import pandas as pd

EXPECTED_PROCESSED_SPLITS: tuple[str, ...] = ("train", "val_model", "val_policy", "test")


def _json_default(value: Any) -> Any:
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, set):
        return sorted(value)
    raise TypeError(f"Object of type {type(value)} is not JSON serializable")


def stable_json_dumps(payload: Any) -> str:
    return json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        default=_json_default,
    )


def sha256_json(payload: Any) -> str:
    return hashlib.sha256(stable_json_dumps(payload).encode("utf-8")).hexdigest()


def column_sequence_hash(columns: Iterable[str]) -> str:
    return sha256_json(list(columns))


def rare_map_schema_hash(rare_map: Mapping[str, set[str]]) -> str:
    return sha256_json({key: sorted(value) for key, value in sorted(rare_map.items())})


def dataframe_schema_hash(df: pd.DataFrame) -> str:
    payload = [{"name": str(col), "dtype": str(df[col].dtype)} for col in df.columns]
    return sha256_json(payload)


def dataframe_fingerprint(
    df: pd.DataFrame,
    columns: Iterable[str] | None = None,
) -> str:
    if columns is None:
        preferred = [
            "SK_ID_CURR",
            "TARGET",
            "ADV_LABEL",
            "DAYS_ID_PUBLISH",
            "DAYS_REGISTRATION",
        ]
        selected = [column for column in preferred if column in df.columns]
        if not selected:
            selected = list(df.columns[: min(len(df.columns), 10)])
    else:
        selected = [column for column in columns if column in df.columns]
        if not selected:
            raise ValueError("No requested fingerprint columns are present in the DataFrame")

    subset = df[selected].copy()
    hashed = pd.util.hash_pandas_object(subset, index=False).to_numpy(dtype=np.uint64)
    payload = {
        "rows": int(len(df)),
        "columns": selected,
        "schema_hash": dataframe_schema_hash(subset),
        "hash_sum_mod64": int(hashed.sum(dtype=np.uint64)),
    }
    return sha256_json(payload)


def build_split_summary(df: pd.DataFrame) -> dict[str, Any]:
    summary: dict[str, Any] = {"rows": int(len(df))}

    if "TARGET" in df.columns:
        target = pd.to_numeric(df["TARGET"], errors="coerce")
        summary["positive_count"] = int(target.fillna(0).sum())
        summary["target_rate"] = float(target.mean()) if len(target) else 0.0

    if "DAYS_EMPLOYED_ANOM" in df.columns:
        anom = pd.to_numeric(df["DAYS_EMPLOYED_ANOM"], errors="coerce")
        summary["days_employed_anom_rate"] = float(anom.fillna(0).mean()) if len(anom) else 0.0

    if "DAYS_ID_PUBLISH" in df.columns:
        recency = pd.to_numeric(df["DAYS_ID_PUBLISH"], errors="coerce").abs()
        summary["mean_abs_days_id_publish"] = float(recency.mean()) if len(recency) else 0.0

    if "DAYS_REGISTRATION" in df.columns:
        recency = pd.to_numeric(df["DAYS_REGISTRATION"], errors="coerce").abs()
        summary["mean_abs_days_registration"] = float(recency.mean()) if len(recency) else 0.0

    return summary


def safe_git_commit(cwd: str | None = None) -> str | None:
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=cwd,
            check=False,
            capture_output=True,
            text=True,
        )
    except OSError:
        return None

    commit = result.stdout.strip()
    return commit or None


def load_json_object(path: str, label: str) -> dict[str, Any]:
    if not os.path.exists(path):
        raise RuntimeError(f"Missing required {label}: {path}")
    try:
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
    except Exception as exc:
        raise RuntimeError(f"Failed to read {label}: {path}") from exc
    if not isinstance(payload, dict):
        raise RuntimeError(f"{label} must contain a JSON object: {path}")
    return payload


def validate_processed_splits(splits: Mapping[str, pd.DataFrame]) -> dict[str, Any]:
    missing = [name for name in EXPECTED_PROCESSED_SPLITS if name not in splits]
    if missing:
        raise ValueError(f"Missing processed splits for verification: {missing}")

    duplicate_summary: dict[str, int] = {}
    split_schema_hashes: dict[str, str] = {}
    split_fingerprints: dict[str, str] = {}
    split_summary: dict[str, dict[str, Any]] = {}

    for split_name in EXPECTED_PROCESSED_SPLITS:
        df = splits[split_name]
        if not isinstance(df, pd.DataFrame):
            raise TypeError(f"{split_name} must be a pandas DataFrame")
        if "SK_ID_CURR" not in df.columns:
            raise ValueError(f"{split_name} is missing SK_ID_CURR")

        dupes = int(df["SK_ID_CURR"].duplicated().sum())
        duplicate_summary[f"{split_name}_duplicate_sk_id_curr"] = dupes
        if dupes:
            raise ValueError(f"{split_name} contains duplicate SK_ID_CURR values")

        split_schema_hashes[split_name] = dataframe_schema_hash(df)
        split_fingerprints[split_name] = dataframe_fingerprint(df)
        split_summary[split_name] = build_split_summary(df)

    all_ids = pd.concat(
        [
            splits[split_name][["SK_ID_CURR"]].assign(__split=split_name)
            for split_name in EXPECTED_PROCESSED_SPLITS
        ],
        ignore_index=True,
    )
    cross_split_dupes = int(all_ids["SK_ID_CURR"].duplicated().sum())
    duplicate_summary["cross_split_duplicate_sk_id_curr"] = cross_split_dupes
    if cross_split_dupes:
        raise ValueError("Processed splits reuse SK_ID_CURR across partitions")

    unique_schema_hashes = set(split_schema_hashes.values())
    if len(unique_schema_hashes) != 1:
        raise ValueError("Processed splits do not share an identical schema")

    for left, right in zip(EXPECTED_PROCESSED_SPLITS, EXPECTED_PROCESSED_SPLITS[1:]):
        left_publish = pd.to_numeric(splits[left]["DAYS_ID_PUBLISH"], errors="coerce").abs().dropna()
        right_publish = pd.to_numeric(splits[right]["DAYS_ID_PUBLISH"], errors="coerce").abs().dropna()
        if not left_publish.empty and not right_publish.empty:
            if float(left_publish.min()) < float(right_publish.max()):
                raise ValueError(
                    "Proxy-time split ordering violated between "
                    f"{left} and {right} on DAYS_ID_PUBLISH"
                )

        left_registration = pd.to_numeric(splits[left]["DAYS_REGISTRATION"], errors="coerce").abs().dropna()
        right_registration = pd.to_numeric(splits[right]["DAYS_REGISTRATION"], errors="coerce").abs().dropna()
        if not left_registration.empty and not right_registration.empty:
            if float(left_registration.mean()) < float(right_registration.mean()):
                raise ValueError(
                    "Proxy-time split ordering violated between "
                    f"{left} and {right} on DAYS_REGISTRATION mean"
                )

    return {
        "duplicate_summary": duplicate_summary,
        "split_schema_hashes": split_schema_hashes,
        "split_fingerprints": split_fingerprints,
        "split_summary": split_summary,
    }


def validate_transformed_frame(
    df: pd.DataFrame,
    *,
    expected_rows: int | None = None,
    expected_columns: Iterable[str] | None = None,
    forbidden_columns: Iterable[str] = (),
    fail_on_constant_columns: bool = False,
) -> dict[str, Any]:
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Builder transform must return a pandas DataFrame")

    if expected_rows is not None and len(df) != expected_rows:
        raise ValueError(f"Expected {expected_rows} rows, got {len(df)}")

    non_numeric = [column for column in df.columns if not pd.api.types.is_numeric_dtype(df[column])]
    if non_numeric:
        raise ValueError(f"Transformed frame contains non-numeric columns: {non_numeric}")

    if expected_columns is not None and list(df.columns) != list(expected_columns):
        raise ValueError("Transformed frame column order does not match the frozen builder contract")

    blocked = [column for column in df.columns if column in set(forbidden_columns)]
    if blocked:
        raise ValueError(f"Forbidden columns leaked into transformed frame: {blocked}")

    all_null_columns = [column for column in df.columns if df[column].isna().all()]
    if all_null_columns:
        raise ValueError(f"Transformed frame contains all-null columns: {all_null_columns}")

    constant_columns: list[str] = []
    if fail_on_constant_columns:
        constant_columns = [column for column in df.columns if df[column].nunique(dropna=False) <= 1]
        if constant_columns:
            raise ValueError(f"Transformed frame contains constant columns: {constant_columns}")

    return {
        "rows": int(len(df)),
        "feature_count": int(df.shape[1]),
        "schema_hash": dataframe_schema_hash(df),
        "all_null_columns": all_null_columns,
        "constant_columns": constant_columns,
    }


def validate_builder_artifact(
    builder: Any,
    *,
    tier: str,
    manifest: Mapping[str, Any] | None,
    processed_manifest: Mapping[str, Any] | None = None,
    strict: bool,
) -> dict[str, Any]:
    from src.feature_engineering import FrozenFeatureBuilder

    normalized_tier = tier.upper()
    if not isinstance(builder, FrozenFeatureBuilder):
        raise TypeError(f"{normalized_tier} builder must be a FrozenFeatureBuilder")
    if builder.tier.upper() != normalized_tier:
        raise ValueError(
            f"{normalized_tier} builder tier mismatch: got {builder.tier!r}"
        )

    warnings: list[str] = []
    if manifest is None:
        if strict:
            raise RuntimeError(f"Missing required {normalized_tier} builder manifest")
        warnings.append(f"{normalized_tier} builder manifest missing; strict lineage checks skipped")
        return {"warnings": warnings}

    manifest_tier = str(manifest.get("builder_tier", "")).upper()
    if manifest_tier != normalized_tier:
        raise RuntimeError(
            f"{normalized_tier} builder manifest tier mismatch: {manifest_tier!r}"
        )

    encoded_count = int(manifest.get("encoded_column_count", -1))
    if encoded_count != len(builder.encoded_columns_):
        raise RuntimeError(
            f"{normalized_tier} builder encoded column count mismatch"
        )

    expected_encoded_hash = column_sequence_hash(builder.encoded_columns_)
    if manifest.get("encoded_column_schema_hash") != expected_encoded_hash:
        raise RuntimeError(
            f"{normalized_tier} builder encoded column schema hash mismatch"
        )

    expected_pre_model_hash = column_sequence_hash(builder.pre_model_columns_)
    if manifest.get("pre_model_column_schema_hash") != expected_pre_model_hash:
        raise RuntimeError(
            f"{normalized_tier} builder pre-model column schema hash mismatch"
        )

    expected_rare_hash = rare_map_schema_hash(builder.rare_category_maps_)
    if manifest.get("rare_map_schema_hash") != expected_rare_hash:
        raise RuntimeError(f"{normalized_tier} builder rare-map schema hash mismatch")

    processed_fingerprint = manifest.get("processed_manifest_fingerprint")
    if strict:
        if processed_manifest is None:
            raise RuntimeError("Strict builder validation requires a processed manifest")
        expected_fingerprint = processed_manifest.get("processed_manifest_fingerprint")
        if not expected_fingerprint:
            raise RuntimeError(
                "Processed manifest is missing processed_manifest_fingerprint"
            )
        if processed_fingerprint != expected_fingerprint:
            raise RuntimeError(
                f"{normalized_tier} builder lineage mismatch with processed manifest"
            )
    elif processed_manifest is not None:
        expected_fingerprint = processed_manifest.get("processed_manifest_fingerprint")
        if expected_fingerprint and processed_fingerprint and processed_fingerprint != expected_fingerprint:
            raise RuntimeError(
                f"{normalized_tier} builder lineage mismatch with processed manifest"
            )

    return {"warnings": warnings}




## Support: Builder Artifacts Loader\n
\n
These helpers load and validate serialized feature builders. They are included here because the API and runtime paths depend on them.\n
\n
Original source: `src/builder_artifacts.py`\n

In [ ]:
"""Canonical builder artifact loader and validator."""

from __future__ import annotations

import os
from typing import Any, Mapping

import joblib

from src.feature_engineering import FrozenFeatureBuilder
from src.runtime_verification import load_json_object, validate_builder_artifact

PROCESSED_MANIFEST_FILENAME = "processed_artifact_manifest.json"
_BUILDER_FILENAMES = {
    "FULL": ("full_feature_builder.joblib", "full_feature_builder.manifest.json"),
    "REDUCED": ("reduced_feature_builder.joblib", "reduced_feature_builder.manifest.json"),
}


def _normalize_tier(tier: str) -> str:
    normalized = str(tier).upper()
    if normalized not in _BUILDER_FILENAMES:
        raise ValueError(f"Unsupported builder tier: {tier!r}")
    return normalized


def _builder_paths(artifact_dir: str, tier: str) -> tuple[str, str]:
    joblib_name, manifest_name = _BUILDER_FILENAMES[_normalize_tier(tier)]
    return os.path.join(artifact_dir, joblib_name), os.path.join(artifact_dir, manifest_name)


def _resolve_processed_manifest(
    processed_dir: str,
    processed_manifest: Mapping[str, Any] | None,
    strict_artifacts: bool,
) -> Mapping[str, Any] | None:
    if processed_manifest is not None:
        return processed_manifest
    manifest_path = os.path.join(processed_dir, PROCESSED_MANIFEST_FILENAME)
    if not os.path.exists(manifest_path):
        if strict_artifacts:
            raise RuntimeError(f"Missing required processed manifest: {manifest_path}")
        return None
    return load_json_object(manifest_path, "processed manifest")


def load_builder(
    *,
    tier: str,
    artifact_dir: str,
    processed_dir: str,
    processed_manifest: Mapping[str, Any] | None = None,
    strict_artifacts: bool = True,
) -> FrozenFeatureBuilder:
    normalized_tier = _normalize_tier(tier)
    builder_path, manifest_path = _builder_paths(artifact_dir, normalized_tier)
    if not os.path.exists(builder_path):
        raise RuntimeError(f"Missing required {normalized_tier} builder artifact: {builder_path}")

    try:
        builder = joblib.load(builder_path)
    except Exception as exc:
        raise RuntimeError(f"Failed to load {normalized_tier} builder artifact: {builder_path}") from exc

    manifest: Mapping[str, Any] | None = None
    if os.path.exists(manifest_path):
        manifest = load_json_object(manifest_path, f"{normalized_tier} builder manifest")
    elif strict_artifacts:
        raise RuntimeError(f"Missing required {normalized_tier} builder manifest: {manifest_path}")

    resolved_processed_manifest = _resolve_processed_manifest(
        processed_dir,
        processed_manifest,
        strict_artifacts,
    )
    validate_builder_artifact(
        builder,
        tier=normalized_tier,
        manifest=manifest,
        processed_manifest=resolved_processed_manifest,
        strict=strict_artifacts,
    )
    return builder


def load_full_builder(
    *,
    artifact_dir: str,
    processed_dir: str,
    processed_manifest: Mapping[str, Any] | None = None,
    strict_artifacts: bool = True,
) -> FrozenFeatureBuilder:
    return load_builder(
        tier="FULL",
        artifact_dir=artifact_dir,
        processed_dir=processed_dir,
        processed_manifest=processed_manifest,
        strict_artifacts=strict_artifacts,
    )


def load_reduced_builder(
    *,
    artifact_dir: str,
    processed_dir: str,
    processed_manifest: Mapping[str, Any] | None = None,
    strict_artifacts: bool = True,
) -> FrozenFeatureBuilder:
    return load_builder(
        tier="REDUCED",
        artifact_dir=artifact_dir,
        processed_dir=processed_dir,
        processed_manifest=processed_manifest,
        strict_artifacts=strict_artifacts,
    )


def load_validated_builders(
    *,
    artifact_dir: str,
    processed_dir: str,
    processed_manifest: Mapping[str, Any] | None = None,
    strict_artifacts: bool = True,
) -> dict[str, FrozenFeatureBuilder]:
    resolved_processed_manifest = _resolve_processed_manifest(
        processed_dir,
        processed_manifest,
        strict_artifacts,
    )
    return {
        "FULL": load_builder(
            tier="FULL",
            artifact_dir=artifact_dir,
            processed_dir=processed_dir,
            processed_manifest=resolved_processed_manifest,
            strict_artifacts=strict_artifacts,
        ),
        "REDUCED": load_builder(
            tier="REDUCED",
            artifact_dir=artifact_dir,
            processed_dir=processed_dir,
            processed_manifest=resolved_processed_manifest,
            strict_artifacts=strict_artifacts,
        ),
    }


## Support: Runtime Support\n
\n
This small helper wraps the tree explainer object used during model runtime.\n
\n
Original source: `src/models/runtime_support.py`\n

In [ ]:
"""Runtime helpers for persisted Module 3 artifacts."""

from __future__ import annotations

from typing import Any

import numpy as np


class TreeShapExplainer:
    """Pickle-friendly SHAP wrapper around a tree model."""

    def __init__(self, model: Any):
        self.model = model
        self._explainer = None

    def _get_explainer(self):
        if self._explainer is None:
            import shap

            self._explainer = shap.TreeExplainer(self.model)
        return self._explainer

    def __call__(self, X: Any):
        arr = np.asarray(X, dtype=float)
        return self._get_explainer()(arr)

    def shap_values(self, X: Any):
        arr = np.asarray(X, dtype=float)
        return self._get_explainer().shap_values(arr)

    def __getstate__(self) -> dict[str, Any]:
        state = dict(self.__dict__)
        state["_explainer"] = None
        return state


## Module 1: Data Pipeline\n
\n
Module 1 handles data loading, cleaning, trap handling, ordered splitting, serialization, and EDA artifact generation.\n
\n
Original source: `src/data_pipeline.py`\n

In [ ]:
"""Module 1 — Data Pipeline.

This module handles:
- Data loading
- Schema enforcement
- Data trap handling (Stage 2)
- Proxy recency sort & ordered split (Stage 3)
"""

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import pickle
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import json


# ================================
# CONSTANTS
# ================================

LOCKED_SCORING_TABLES: set[str] = {
    "application_train.csv",
    "bureau.csv",
    "previous_application.csv",
    "installments_payments.csv",
    "POS_CASH_balance.csv",
    "credit_card_balance.csv"
}

ADVERSARIAL_ONLY_TABLES: set[str] = {
    "application_test.csv"
}

TRAIN_SCHEMA: dict[str, str] = {
    "SK_ID_CURR": "int64",
    "CNT_CHILDREN": "int64",
    "CNT_FAM_MEMBERS": "float64",
    "AMT_INCOME_TOTAL": "float64",
    "AMT_CREDIT": "float64",
    "AMT_ANNUITY": "float64",
    "AMT_GOODS_PRICE": "float64",
    "DAYS_BIRTH": "float64",
    "DAYS_EMPLOYED": "float64",
    "DAYS_REGISTRATION": "float64",
    "DAYS_ID_PUBLISH": "float64",
    "DAYS_LAST_PHONE_CHANGE": "float64",
    "EXT_SOURCE_1": "float64",
    "EXT_SOURCE_2": "float64",
    "EXT_SOURCE_3": "float64",
    "NAME_CONTRACT_TYPE": "object",
    "NAME_TYPE_SUITE": "object",
    "NAME_EDUCATION_TYPE": "object",
    "NAME_FAMILY_STATUS": "object",
    "OCCUPATION_TYPE": "object",
    "ORGANIZATION_TYPE": "object",
    "WEEKDAY_APPR_PROCESS_START": "object",
    "NAME_INCOME_TYPE": "object",
    "NAME_HOUSING_TYPE": "object",
    "FLAG_OWN_CAR": "object",
    "FLAG_OWN_REALTY": "object",
    "REGION_RATING_CLIENT_W_CITY": "float64"
}

PREV_SENTINEL_DAY_COLS: list[str] = [
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION"
]

# All DAYS_* and MONTHS_BALANCE fields are relative offsets,
# not calendar timestamps.
RELATIVE_TIME_COLS: set[str] = {
    "DAYS_BIRTH", "DAYS_EMPLOYED", "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH", "DAYS_DECISION", "MONTHS_BALANCE",
    "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT"
}

# ================================
# FUNCTIONS
# ================================

def enforce_locked_tables(loaded_names: list[str]) -> None:
    unexpected = set(loaded_names) - LOCKED_SCORING_TABLES - ADVERSARIAL_ONLY_TABLES
    if unexpected:
        raise ValueError(f"Unexpected table(s) referenced: {sorted(unexpected)}")


def enforce_schema(df: pd.DataFrame, schema: dict[str, str]) -> pd.DataFrame:
    df_copy = df.copy()
    for col, dtype in schema.items():
        if col in df_copy.columns:
            df_copy[col] = df_copy[col].astype(np.dtype(dtype))
    return df_copy


# ================================
# TRAP D — MISSING POLICY FUNCTIONS
# ================================

def fit_missing_policy(train_df: pd.DataFrame):
    miss_rate = train_df.isna().mean()
    flag_cols = miss_rate[miss_rate >= 0.05].index.tolist()

    numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = train_df.select_dtypes(exclude=[np.number]).columns.tolist()

    num_medians = train_df[
        [c for c in numeric_cols if c in flag_cols]
    ].median(numeric_only=True).to_dict()

    cat_modes = {
        c: (
            train_df[c].mode(dropna=True).iloc[0]
            if train_df[c].dropna().shape[0] > 0
            else "MISSING"
        )
        for c in categorical_cols if c in flag_cols
    }

    return (flag_cols, num_medians, cat_modes)


def apply_missing_policy(
    df: pd.DataFrame,
    flag_cols: list[str],
    num_medians: dict[str, float],
    cat_modes: dict[str, str],
    for_linear_model: bool = False
) -> pd.DataFrame:

    df = df.copy()

    for col in flag_cols:
        if col in df.columns:
            df[f"{col}_IS_MISSING"] = df[col].isna().astype("int8")

    if for_linear_model:
        for col, val in num_medians.items():
            if col in df.columns:
                df[col] = df[col].fillna(val)

        for col, val in cat_modes.items():
            if col in df.columns:
                df[col] = df[col].fillna(val)

    return df


# ================================
# STAGE 3 FUNCTIONS
# ================================

# ordered holdout using Home Credit recency proxies;
# not true calendar-time validation
def proxy_recency_sort(app_df: pd.DataFrame) -> pd.DataFrame:
    tmp = app_df.copy()

    tmp["__RECENCY_1"] = tmp["DAYS_ID_PUBLISH"].abs().fillna(
        tmp["DAYS_ID_PUBLISH"].abs().median()
    )
    tmp["__RECENCY_2"] = tmp["DAYS_REGISTRATION"].abs().fillna(
        tmp["DAYS_REGISTRATION"].abs().median()
    )

    tmp = tmp.sort_values(
        ["__RECENCY_1", "__RECENCY_2", "SK_ID_CURR"],
        ascending=[True, True, True]
    )

    tmp = tmp.drop(columns=["__RECENCY_1", "__RECENCY_2"])
    return tmp.reset_index(drop=True)


def ordered_split_60_10_10_20(df: pd.DataFrame):
    n = len(df)

    i60 = int(n * 0.60)
    i70 = int(n * 0.70)
    i80 = int(n * 0.80)

    train = df.iloc[:i60].copy()
    val_model = df.iloc[i60:i70].copy()
    val_policy = df.iloc[i70:i80].copy()
    test = df.iloc[i80:].copy()

    return (train, val_model, val_policy, test)


def build_adversarial_dataset(train_app: pd.DataFrame, test_app: pd.DataFrame):

    # Step 1: Balance dataset
    min_size = min(len(train_app), len(test_app))

    train_sample = train_app.sample(n=min_size, random_state=42)
    test_sample = test_app.sample(n=min_size, random_state=42)

    # Step 2: Assign labels
    adv_train = train_sample.copy()
    adv_train["ADV_LABEL"] = 0

    adv_test = test_sample.copy()
    adv_test["ADV_LABEL"] = 1

    # Step 3: Combine
    combined = pd.concat([adv_train, adv_test], ignore_index=True)

    # Step 4: Split
    adv_X_train, adv_X_val = train_test_split(
        combined,
        test_size=0.20,
        stratify=combined["ADV_LABEL"],
        random_state=42
    )

    return (
        adv_X_train.reset_index(drop=True),
        adv_X_val.reset_index(drop=True)
    )


# ================================
# MAIN EXECUTION BLOCK
# ================================

if __name__ == "__main__":

    DATA_RAW_DIR = os.environ.get("DATA_RAW_DIR", "data/raw/")

    filenames = [
        "application_train.csv",
        "bureau.csv",
        "previous_application.csv",
        "installments_payments.csv",
        "POS_CASH_balance.csv",
        "credit_card_balance.csv",
        "application_test.csv"
    ]

    # Load datasets
    app_train = pd.read_csv(os.path.join(DATA_RAW_DIR, filenames[0]))
    bureau = pd.read_csv(os.path.join(DATA_RAW_DIR, filenames[1]))
    prev_app = pd.read_csv(os.path.join(DATA_RAW_DIR, filenames[2]))
    inst = pd.read_csv(os.path.join(DATA_RAW_DIR, filenames[3]))
    pos_cash = pd.read_csv(os.path.join(DATA_RAW_DIR, filenames[4]))
    cc = pd.read_csv(os.path.join(DATA_RAW_DIR, filenames[5]))
    app_test = pd.read_csv(os.path.join(DATA_RAW_DIR, filenames[6]))

    # Enforce table whitelist
    enforce_locked_tables(filenames)

    # Apply schema
    app_train = enforce_schema(app_train, TRAIN_SCHEMA)

    # ================================
    # TRAP A — DAYS_EMPLOYED FIX
    # ================================
    app_train["DAYS_EMPLOYED_ANOM"] = (
        app_train["DAYS_EMPLOYED"] == 365243
    ).astype("int8")

    app_train["DAYS_EMPLOYED"] = app_train["DAYS_EMPLOYED"].replace(365243, np.nan)

    # ================================
    # TRAP B — PREV_APP FIX
    # ================================
    for col in PREV_SENTINEL_DAY_COLS:
        if col in prev_app.columns:
            prev_app[f"{col}_ANOM"] = (prev_app[col] == 365243).astype("int8")
            prev_app[col] = prev_app[col].replace(365243, np.nan)

    # ================================
    # TRAP E — SCHEMA ASSERTION
    # ================================
    assert app_train["SK_ID_CURR"].dtype == np.int64, \
        "SK_ID_CURR dtype enforcement failed"

    # ================================
    # STAGE 3 — SORT + SPLIT
    # ================================
    app_sorted = proxy_recency_sort(app_train)

    train, val_model, val_policy, test = ordered_split_60_10_10_20(app_sorted)

    # ================================
    # TRAP C — INCOME CAP
    # ================================
    income_cap = train["AMT_INCOME_TOTAL"].quantile(0.99)

    for partition in [train, val_model, val_policy, test]:
        partition["AMT_INCOME_TOTAL_CAPPED"] = partition["AMT_INCOME_TOTAL"].clip(
            upper=income_cap
        )

    # ================================
    # ADVERSARIAL DATASET
    # ================================
    adv_train_df, adv_val_df = build_adversarial_dataset(app_train, app_test)

    # ================================
    # PRINT SHAPES
    # ================================
    print(f"{filenames[0]}: {app_train.shape[0]} rows, {app_train.shape[1]} cols")
    print(f"{filenames[1]}: {bureau.shape[0]} rows, {bureau.shape[1]} cols")
    print(f"{filenames[2]}: {prev_app.shape[0]} rows, {prev_app.shape[1]} cols")
    print(f"{filenames[3]}: {inst.shape[0]} rows, {inst.shape[1]} cols")
    print(f"{filenames[4]}: {pos_cash.shape[0]} rows, {pos_cash.shape[1]} cols")
    print(f"{filenames[5]}: {cc.shape[0]} rows, {cc.shape[1]} cols")
    print(f"{filenames[6]}: {app_test.shape[0]} rows, {app_test.shape[1]} cols")

    print("\nSplit Sizes:")
    print(f"train:      {len(train)} rows")
    print(f"val_model:  {len(val_model)} rows")
    print(f"val_policy: {len(val_policy)} rows")
    print(f"test:       {len(test)} rows")
    print(f"adv_train:  {len(adv_train_df)} rows")
    print(f"adv_val:    {len(adv_val_df)} rows")

    # ================================
    # VALIDATION CHECKS (STAGE 3)
    # ================================
    assert len(train) + len(val_model) + len(val_policy) + len(test) == len(app_train)

    ratio = len(train) / len(app_train)
    assert 0.599 <= ratio <= 0.601

    assert train.index[0] == 0

    for df in [train, val_model, val_policy, test]:
        assert "AMT_INCOME_TOTAL_CAPPED" in df.columns

    assert adv_train_df["ADV_LABEL"].nunique() == 2

    val_dist = adv_val_df["ADV_LABEL"].value_counts(normalize=True)
    assert abs(val_dist[0] - 0.5) < 0.05
    
    # ================================
    # STAGE 4 — DIRECTORY SETUP
    # ================================

    DATA_PROCESSED_DIR = os.environ.get(
        "DATA_PROCESSED_DIR", "data/processed/"
    )

    os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)

    # ================================
    # SERIALIZATION FUNCTION
    # ================================

    def serialize_dataframe(df: pd.DataFrame, path: str) -> None:
        with open(path, "wb") as f:
            pickle.dump(df, f, protocol=4)
        print(f"Saved {path}: {df.shape[0]} rows, {df.shape[1]} cols")

    # ================================
    # SAVE DATAFRAMES
    # ================================

    serialize_dataframe(train, DATA_PROCESSED_DIR + "train.pkl")
    serialize_dataframe(val_model, DATA_PROCESSED_DIR + "val_model.pkl")
    serialize_dataframe(val_policy, DATA_PROCESSED_DIR + "val_policy.pkl")
    serialize_dataframe(test, DATA_PROCESSED_DIR + "test.pkl")

    adv_path = DATA_PROCESSED_DIR + "app_test_adv.pkl"
    with open(adv_path, "wb") as f:
        pickle.dump(
            {"adv_train": adv_train_df, "adv_val": adv_val_df},
            f,
            protocol=4
        )
    print(f"Saved {adv_path}")

    # ================================
    # SAVE INCOME CAP
    # ================================

    joblib.dump(income_cap, DATA_PROCESSED_DIR + "income_cap.joblib")
    print(f"income_cap = {income_cap:.2f} saved")

    # ================================
    # RELOAD VERIFICATION
    # ================================

    def verify_dataframe(path: str, original_df: pd.DataFrame):
        with open(path, "rb") as f:
            reloaded = pickle.load(f)

        assert isinstance(reloaded, pd.DataFrame), \
            f"{path} did not reload as DataFrame"

        assert reloaded.shape == original_df.shape, \
            f"{path} shape mismatch after reload"


    verify_dataframe(DATA_PROCESSED_DIR + "train.pkl", train)
    verify_dataframe(DATA_PROCESSED_DIR + "val_model.pkl", val_model)
    verify_dataframe(DATA_PROCESSED_DIR + "val_policy.pkl", val_policy)
    verify_dataframe(DATA_PROCESSED_DIR + "test.pkl", test)

    with open(adv_path, "rb") as f:
        adv_loaded = pickle.load(f)

    assert isinstance(adv_loaded, dict)
    assert set(adv_loaded.keys()) == {"adv_train", "adv_val"}

    loaded_income_cap = joblib.load(DATA_PROCESSED_DIR + "income_cap.joblib")
    assert isinstance(loaded_income_cap, float)

    print("\nAll serialization checks passed.")
    
    # ================================
    # STAGE 5 — EDA DIRECTORY
    # ================================

    EDA_PLOTS_DIR = os.environ.get(
        "EDA_PLOTS_DIR", "notebooks/eda_plots/"
    )

    os.makedirs(EDA_PLOTS_DIR, exist_ok=True)
    
    # Plot 1 — Target Distribution
    
    fig, ax = plt.subplots(figsize=(6, 4))
    counts = train["TARGET"].value_counts().sort_index()

    ax.bar(
    ["No default (0)", "Default (1)"],
    counts.values,
    color=["#1D9E75", "#D85A30"]
    )

    ax.set_title("Target distribution (train partition)")
    ax.set_ylabel("Count")

    for i, v in enumerate(counts.values):
        ax.text(i, v, f"{v:,} ({v/len(train)*100:.1f}%)",
            fontsize=10, ha="center", va="bottom")

    plt.tight_layout()
    plt.savefig(EDA_PLOTS_DIR + "target_distribution.png", dpi=150)
    plt.close()
    
    # Plot 2 — Missing Heatmap
    
    miss_rate = app_train.isna().mean().sort_values(ascending=False)
    miss_top = miss_rate[miss_rate > 0].head(40)

    fig, ax = plt.subplots(figsize=(10, 8))

    sns.heatmap(
        miss_top.to_frame().T,
        annot=False,
        cmap="YlOrRd",
        ax=ax,
        vmin=0,
        vmax=1,
        cbar_kws={"label": "Missing rate"}
    )

    ax.set_title("Missing value rates — application_train (top 40)")
    ax.set_xlabel("Column")
    plt.xticks(rotation=90, fontsize=7)

    plt.tight_layout()
    plt.savefig(EDA_PLOTS_DIR + "missing_value_heatmap.png", dpi=150)
    plt.close()
    
    #Plot 3 — Correlation
    cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "TARGET"]
    sub = train[cols].dropna()
    corr = sub.corr()

    fig, ax = plt.subplots(figsize=(5, 4))

    sns.heatmap(
        corr,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        ax=ax,
        vmin=-1,
        vmax=1
    )

    ax.set_title("EXT_SOURCE & TARGET correlations (train)")

    plt.tight_layout()
    plt.savefig(EDA_PLOTS_DIR + "ext_source_correlation.png", dpi=150)
    plt.close()
    
    #Plot 4 — DAYS_EMPLOYED anomaly
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    raw_vals = app_train["DAYS_EMPLOYED"].copy()
    raw_vals_with_sentinel = raw_vals.copy()

    raw_vals_with_sentinel[
        app_train["DAYS_EMPLOYED_ANOM"] == 1
    ] = 365243

    axes[0].hist(
        raw_vals_with_sentinel.dropna(),
        bins=50,
        color="#378ADD",
        edgecolor="none"
    )

    axes[0].set_title("DAYS_EMPLOYED — with sentinel (365243)")

    axes[1].hist(
        app_train["DAYS_EMPLOYED"].dropna(),
        bins=50,
        color="#1D9E75",
        edgecolor="none"
    )

    axes[1].set_title("DAYS_EMPLOYED — after Trap A fix")

    for ax in axes:
        ax.set_xlabel("Value")
        ax.set_ylabel("Count")

    plt.suptitle("Trap A: DAYS_EMPLOYED sentinel removal")

    plt.tight_layout()
    plt.savefig(EDA_PLOTS_DIR + "days_employed_anomaly.png", dpi=150)
    plt.close()
    
    #Plot 5 — Income Outliers
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(
        train["AMT_INCOME_TOTAL"].clip(upper=2e6).dropna(),
        bins=60,
        color="#7F77DD",
        edgecolor="none"
    )

    axes[0].set_title("AMT_INCOME_TOTAL — raw (clipped at 2M)")

    axes[1].hist(
        train["AMT_INCOME_TOTAL_CAPPED"].dropna(),
        bins=60,
        color="#1D9E75",
        edgecolor="none"
    )

    axes[1].axvline(
        x=income_cap,
        color="#D85A30",
        linestyle="--",
        linewidth=1.5,
        label=f"99th pct = {income_cap:,.0f}"
    )

    axes[1].legend(fontsize=9)
    axes[1].set_title("AMT_INCOME_TOTAL — after Trap C cap")

    for ax in axes:
        ax.set_xlabel("Value")
        ax.set_ylabel("Count")

    plt.suptitle("Trap C: Income outlier capping")

    plt.tight_layout()
    plt.savefig(EDA_PLOTS_DIR + "income_outliers.png", dpi=150)
    plt.close()
    
    # Data Quality Report
    
    report = {
        "dataset": "application_train",
        "total_rows": int(len(app_train)),
        "total_cols": int(len(app_train.columns)),
        "target_default_rate": float(train["TARGET"].mean().round(4)),
        "class_counts": {
            str(k): int(v)
            for k, v in train["TARGET"].value_counts().items()
        },
        "sentinel_counts": {
            "DAYS_EMPLOYED_365243":
                int(app_train["DAYS_EMPLOYED_ANOM"].sum())
        },
        "missing_rates": {
            col: float(round(rate, 4))
            for col, rate in app_train.isna().mean().items()
            if rate > 0
        },
        "income_cap_p99": float(round(income_cap, 2)),
        "split_sizes": {
            "train": int(len(train)),
            "val_model": int(len(val_model)),
            "val_policy": int(len(val_policy)),
            "test": int(len(test))
        }
    }

    with open("data/data_quality_report.json", "w") as f:
        json.dump(report, f, indent=2)

    print("data_quality_report.json saved")


## Module 2: Feature Engineering\n
\n
Module 2 creates engineered application features, aggregate features, builder fitting logic, and transform-time encoding logic.\n
\n
Original source: `src/feature_engineering.py`\n

In [ ]:
"""MasterMind Module 2 - Feature Engineering."""

from __future__ import annotations

import os
from dataclasses import dataclass, field

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

CATEGORICAL_MODEL_COLS = [
    "NAME_CONTRACT_TYPE",
    "NAME_TYPE_SUITE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
    "WEEKDAY_APPR_PROCESS_START",
]
NUMERIC_RAW_COLS = [
    "AMT_INCOME_TOTAL_CAPPED",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "DAYS_LAST_PHONE_CHANGE",
    "REGION_POPULATION_RELATIVE",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "CNT_FAM_MEMBERS",
    "OWN_CAR_AGE",
    "OBS_30_CNT_SOCIAL_CIRCLE",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE",
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
]
FAIRNESS_ONLY_COLS = [
    "REGION_RATING_CLIENT_W_CITY",
    "NAME_INCOME_TYPE",
    "NAME_HOUSING_TYPE",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "CNT_CHILDREN",
]
FORBIDDEN_COLS = ["SK_ID_CURR", "TARGET", "CODE_GENDER", "SK_ID_PREV"]
APPLICATION_REQUIRED_INPUT_COLS = NUMERIC_RAW_COLS + CATEGORICAL_MODEL_COLS + [
    "DAYS_EMPLOYED_ANOM"
]
ENGINEERED_APP_FEATURE_COLS = [
    "AGE_YEARS",
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "GOODS_CREDIT_RATIO",
    "CREDIT_TERM_RATIO",
    "EMPLOYED_BIRTH_RATIO",
    "ID_PUBLISH_REG_RATIO",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_STD",
    "SOCIAL_CIRCLE_SUM",
    "BUREAU_REQUEST_SUM",
    "DAYS_EMPLOYED_ANOM",
]
BUREAU_AGG_COLS = [
    "BUREAU_LOAN_COUNT",
    "BUREAU_ACTIVE_COUNT",
    "BUREAU_CLOSED_COUNT",
    "BUREAU_AMT_CREDIT_SUM_SUM",
    "BUREAU_AMT_CREDIT_SUM_DEBT_SUM",
    "BUREAU_DEBT_TO_CREDIT_RATIO",
    "BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM",
    "BUREAU_CREDIT_DAY_OVERDUE_MAX",
    "BUREAU_DAYS_CREDIT_MAX",
    "BUREAU_CNT_CREDIT_PROLONG_SUM",
]
PREVIOUS_AGG_COLS = [
    "PREV_APP_COUNT",
    "PREV_APPROVED_COUNT",
    "PREV_REFUSED_COUNT",
    "PREV_APPROVAL_RATE",
    "PREV_REFUSAL_RATE",
    "PREV_AMT_APPLICATION_MEAN",
    "PREV_AMT_CREDIT_MEAN",
    "PREV_AMT_GOODS_PRICE_MEAN",
    "PREV_APP_CREDIT_DIFF_MEAN",
    "PREV_DAYS_DECISION_MAX",
    "PREV_RATE_DOWN_PAYMENT_MEAN",
]
INSTALLMENTS_AGG_COLS = [
    "INST_RECORD_COUNT",
    "INST_MISSED_RATE",
    "INST_DPD_MEAN",
    "INST_DPD_MAX",
    "INST_PAYMENT_RATIO_MEAN",
    "INST_PAYMENT_RATIO_MIN",
    "INST_LATE_COUNT",
]
POS_CASH_AGG_COLS = [
    "POS_RECORD_COUNT",
    "POS_DPD_MEAN",
    "POS_DPD_MAX",
    "POS_DPD_DEF_MEAN",
    "POS_DPD_DEF_MAX",
    "POS_COMPLETED_RATE",
    "POS_ACTIVE_RATE",
    "POS_CNT_INSTALMENT_FUTURE_MEAN",
]
CREDIT_CARD_AGG_COLS = [
    "CC_RECORD_COUNT",
    "CC_BALANCE_MEAN",
    "CC_LIMIT_MEAN",
    "CC_UTILIZATION_MEAN",
    "CC_PAYMENT_RATIO_MEAN",
    "CC_DPD_MEAN",
    "CC_DPD_MAX",
    "CC_DRAWINGS_ATM_SUM",
    "CC_DRAWINGS_CURRENT_SUM",
]
ALL_AGGREGATE_FEATURE_COLS = (
    BUREAU_AGG_COLS
    + PREVIOUS_AGG_COLS
    + INSTALLMENTS_AGG_COLS
    + POS_CASH_AGG_COLS
    + CREDIT_CARD_AGG_COLS
)
FULL_FEATURE_BUILDER_ARTIFACT_PATH = "artifacts/full_feature_builder.joblib"
REDUCED_FEATURE_BUILDER_ARTIFACT_PATH = "artifacts/reduced_feature_builder.joblib"
PROCESSED_SPLIT_NAMES = ["train", "val_model", "val_policy", "test"]
__all__ = [
    "FrozenFeatureBuilder",
    "fit_full_builder",
    "fit_reduced_builder",
    "build_full",
    "build_reduced",
    "pool_rare_categories",
    "safe_div",
    "assert_unique_key",
    "safe_left_merge_one_to_one",
    "prefix_columns",
]


@dataclass
class FrozenFeatureBuilder:
    tier: str
    rare_category_maps_: dict[str, set[str]] = field(default_factory=dict)
    flag_columns_: list[str] = field(default_factory=list)
    numeric_imputers_: dict[str, float] = field(default_factory=dict)
    categorical_fill_values_: dict[str, str] = field(default_factory=dict)
    pre_model_columns_: list[str] = field(default_factory=list)
    encoded_columns_: list[str] = field(default_factory=list)
    numeric_scale_columns_: list[str] = field(default_factory=list)
    aggregate_feature_cols_: list[str] = field(default_factory=list)
    categorical_columns_: list[str] = field(default_factory=list)
    scaler_: StandardScaler | None = None

    def save(self, path: str) -> None:
        directory = os.path.dirname(path)
        if directory:
            os.makedirs(directory, exist_ok=True)
        joblib.dump(self, path)

    def transform(
        self,
        df: pd.DataFrame,
        for_linear_model: bool = False,
        raw_dir: str | None = None,
    ) -> pd.DataFrame:
        return _transform_with_builder(df, self, for_linear_model, raw_dir)


def pool_rare_categories(series: pd.Series, min_count: int = 500) -> pd.Series:
    vc = series.value_counts(dropna=False)
    keep = vc[vc >= min_count].index
    return series.where(series.isin(keep), other="OTHER")


def safe_div(a: pd.Series | np.ndarray, b: pd.Series | np.ndarray) -> np.ndarray:
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where((pd.notna(a)) & (pd.notna(b)) & (b != 0), a / b, np.nan)


def assert_unique_key(df: pd.DataFrame, key: str, name: str) -> None:
    dupes = df[key].duplicated().sum()
    if dupes > 0:
        raise ValueError(f"{name} is not unique on {key}: {dupes} duplicate keys")


def safe_left_merge_one_to_one(
    base_df: pd.DataFrame,
    feat_df: pd.DataFrame,
    key: str = "SK_ID_CURR",
    feat_name: str = "features",
) -> pd.DataFrame:
    assert_unique_key(base_df, key, "base_df")
    assert_unique_key(feat_df, key, feat_name)
    return base_df.merge(feat_df, on=key, how="left", validate="one_to_one")


def prefix_columns(df: pd.DataFrame, prefix: str, key: str = "SK_ID_CURR") -> pd.DataFrame:
    cols = [c for c in df.columns if c != key]
    return df.rename(columns={c: f"{prefix}_{c}".upper() for c in cols})


def _engineer_application_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    assert "AMT_INCOME_TOTAL_CAPPED" in df.columns, "AMT_INCOME_TOTAL_CAPPED missing - Module 1 not applied"
    assert "DAYS_EMPLOYED_ANOM" in df.columns, "DAYS_EMPLOYED_ANOM missing - Module 1 Trap A not applied"
    df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365
    df["CREDIT_INCOME_RATIO"] = safe_div(df["AMT_CREDIT"], df["AMT_INCOME_TOTAL_CAPPED"])
    df["ANNUITY_INCOME_RATIO"] = safe_div(df["AMT_ANNUITY"], df["AMT_INCOME_TOTAL_CAPPED"])
    df["GOODS_CREDIT_RATIO"] = safe_div(df["AMT_GOODS_PRICE"], df["AMT_CREDIT"])
    df["CREDIT_TERM_RATIO"] = safe_div(df["AMT_ANNUITY"], df["AMT_CREDIT"])
    df["EMPLOYED_BIRTH_RATIO"] = safe_div(df["DAYS_EMPLOYED"], df["DAYS_BIRTH"])
    df["ID_PUBLISH_REG_RATIO"] = safe_div(df["DAYS_ID_PUBLISH"], df["DAYS_REGISTRATION"])
    df["EXT_SOURCE_MEAN"] = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
    df["EXT_SOURCE_STD"] = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].std(axis=1)
    df["SOCIAL_CIRCLE_SUM"] = (
        df["OBS_30_CNT_SOCIAL_CIRCLE"].fillna(0)
        + df["DEF_30_CNT_SOCIAL_CIRCLE"].fillna(0)
        + df["OBS_60_CNT_SOCIAL_CIRCLE"].fillna(0)
        + df["DEF_60_CNT_SOCIAL_CIRCLE"].fillna(0)
    )
    req_cols = [
        "AMT_REQ_CREDIT_BUREAU_HOUR",
        "AMT_REQ_CREDIT_BUREAU_DAY",
        "AMT_REQ_CREDIT_BUREAU_WEEK",
        "AMT_REQ_CREDIT_BUREAU_MON",
        "AMT_REQ_CREDIT_BUREAU_QRT",
        "AMT_REQ_CREDIT_BUREAU_YEAR",
    ]
    df["BUREAU_REQUEST_SUM"] = df[req_cols].fillna(0).sum(axis=1)
    return df


def _agg_bureau(raw_dir: str) -> pd.DataFrame:
    bureau = pd.read_csv(os.path.join(raw_dir, "bureau.csv"))
    g = bureau.groupby("SK_ID_CURR")
    out = pd.DataFrame(index=g.size().index)
    out["BUREAU_LOAN_COUNT"] = g.size()
    out["BUREAU_ACTIVE_COUNT"] = g["CREDIT_ACTIVE"].apply(lambda s: (s == "Active").sum())
    out["BUREAU_CLOSED_COUNT"] = g["CREDIT_ACTIVE"].apply(lambda s: (s == "Closed").sum())
    out["BUREAU_AMT_CREDIT_SUM_SUM"] = g["AMT_CREDIT_SUM"].sum()
    out["BUREAU_AMT_CREDIT_SUM_DEBT_SUM"] = g["AMT_CREDIT_SUM_DEBT"].sum()
    out["BUREAU_DEBT_TO_CREDIT_RATIO"] = safe_div(
        out["BUREAU_AMT_CREDIT_SUM_DEBT_SUM"].to_numpy(),
        out["BUREAU_AMT_CREDIT_SUM_SUM"].to_numpy(),
    )
    out["BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM"] = g["AMT_CREDIT_SUM_OVERDUE"].sum()
    out["BUREAU_CREDIT_DAY_OVERDUE_MAX"] = g["CREDIT_DAY_OVERDUE"].max()
    out["BUREAU_DAYS_CREDIT_MAX"] = g["DAYS_CREDIT"].max()
    out["BUREAU_CNT_CREDIT_PROLONG_SUM"] = g["CNT_CREDIT_PROLONG"].sum()
    result = out.reset_index()
    assert_unique_key(result, "SK_ID_CURR", "bureau_agg")
    return result


def _agg_previous(raw_dir: str) -> pd.DataFrame:
    prev = pd.read_csv(os.path.join(raw_dir, "previous_application.csv")).copy()
    prev["PREV_APP_CREDIT_DIFF_ROW"] = prev["AMT_APPLICATION"] - prev["AMT_CREDIT"]
    g = prev.groupby("SK_ID_CURR")
    app_count = g.size()
    approved = g["NAME_CONTRACT_STATUS"].apply(lambda s: (s == "Approved").sum())
    refused = g["NAME_CONTRACT_STATUS"].apply(lambda s: (s == "Refused").sum())
    out = pd.DataFrame(index=app_count.index)
    out["PREV_APP_COUNT"] = app_count
    out["PREV_APPROVED_COUNT"] = approved
    out["PREV_REFUSED_COUNT"] = refused
    out["PREV_APPROVAL_RATE"] = safe_div(approved.to_numpy(), app_count.to_numpy())
    out["PREV_REFUSAL_RATE"] = safe_div(refused.to_numpy(), app_count.to_numpy())
    out["PREV_AMT_APPLICATION_MEAN"] = g["AMT_APPLICATION"].mean()
    out["PREV_AMT_CREDIT_MEAN"] = g["AMT_CREDIT"].mean()
    out["PREV_AMT_GOODS_PRICE_MEAN"] = g["AMT_GOODS_PRICE"].mean()
    out["PREV_APP_CREDIT_DIFF_MEAN"] = g["PREV_APP_CREDIT_DIFF_ROW"].mean()
    out["PREV_DAYS_DECISION_MAX"] = g["DAYS_DECISION"].max()
    out["PREV_RATE_DOWN_PAYMENT_MEAN"] = g["RATE_DOWN_PAYMENT"].mean()
    result = out.reset_index()
    assert_unique_key(result, "SK_ID_CURR", "previous_app_agg")
    return result


def _normalize_child_merge_curr(df: pd.DataFrame, label: str) -> pd.DataFrame:
    if {"SK_ID_CURR_x", "SK_ID_CURR_y"}.issubset(df.columns):
        if not df["SK_ID_CURR_x"].equals(df["SK_ID_CURR_y"]):
            raise ValueError(f"{label} merge produced mismatched SK_ID_CURR values")
        df["SK_ID_CURR"] = df["SK_ID_CURR_y"]
        df = df.drop(columns=["SK_ID_CURR_x", "SK_ID_CURR_y"])
    return df


def _agg_installments(raw_dir: str) -> pd.DataFrame:
    inst = pd.read_csv(os.path.join(raw_dir, "installments_payments.csv"))
    prev_keys = pd.read_csv(
        os.path.join(raw_dir, "previous_application.csv"),
        usecols=["SK_ID_PREV", "SK_ID_CURR"],
    ).drop_duplicates()
    inst = inst.merge(prev_keys, on="SK_ID_PREV", how="inner", validate="many_to_one")
    inst = _normalize_child_merge_curr(inst, "installments").copy()
    inst["INST_MISSED_FLAG"] = inst["DAYS_ENTRY_PAYMENT"].isna().astype("int8")
    inst["INST_DPD"] = np.where(
        inst["DAYS_ENTRY_PAYMENT"].notna(),
        np.maximum(inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"], 0),
        np.nan,
    )
    inst["INST_PAYMENT_RATIO"] = np.where(
        (inst["AMT_INSTALMENT"] > 0) & inst["AMT_PAYMENT"].notna(),
        safe_div(inst["AMT_PAYMENT"], inst["AMT_INSTALMENT"]),
        np.nan,
    )
    g = inst.groupby("SK_ID_CURR")
    out = pd.DataFrame(index=g.size().index)
    out["INST_RECORD_COUNT"] = g.size()
    out["INST_MISSED_RATE"] = g["INST_MISSED_FLAG"].mean()
    out["INST_DPD_MEAN"] = g["INST_DPD"].mean()
    out["INST_DPD_MAX"] = g["INST_DPD"].max()
    out["INST_PAYMENT_RATIO_MEAN"] = g["INST_PAYMENT_RATIO"].mean()
    out["INST_PAYMENT_RATIO_MIN"] = g["INST_PAYMENT_RATIO"].min()
    out["INST_LATE_COUNT"] = g["INST_DPD"].apply(lambda s: (s > 0).sum())
    result = out.reset_index()
    assert_unique_key(result, "SK_ID_CURR", "installments_agg")
    return result


def _agg_pos_cash(raw_dir: str) -> pd.DataFrame:
    pos = pd.read_csv(os.path.join(raw_dir, "POS_CASH_balance.csv"))
    prev_keys = pd.read_csv(
        os.path.join(raw_dir, "previous_application.csv"),
        usecols=["SK_ID_PREV", "SK_ID_CURR"],
    ).drop_duplicates()
    pos = pos.merge(prev_keys, on="SK_ID_PREV", how="inner", validate="many_to_one")
    pos = _normalize_child_merge_curr(pos, "pos cash")
    g = pos.groupby("SK_ID_CURR")
    out = pd.DataFrame(index=g.size().index)
    out["POS_RECORD_COUNT"] = g.size()
    out["POS_DPD_MEAN"] = g["SK_DPD"].mean()
    out["POS_DPD_MAX"] = g["SK_DPD"].max()
    out["POS_DPD_DEF_MEAN"] = g["SK_DPD_DEF"].mean()
    out["POS_DPD_DEF_MAX"] = g["SK_DPD_DEF"].max()
    out["POS_COMPLETED_RATE"] = g["NAME_CONTRACT_STATUS"].apply(lambda s: (s == "Completed").mean())
    out["POS_ACTIVE_RATE"] = g["NAME_CONTRACT_STATUS"].apply(lambda s: (s == "Active").mean())
    out["POS_CNT_INSTALMENT_FUTURE_MEAN"] = g["CNT_INSTALMENT_FUTURE"].mean()
    result = out.reset_index()
    assert_unique_key(result, "SK_ID_CURR", "pos_cash_agg")
    return result


def _agg_credit_card(raw_dir: str) -> pd.DataFrame:
    cc = pd.read_csv(os.path.join(raw_dir, "credit_card_balance.csv"))
    prev_keys = pd.read_csv(
        os.path.join(raw_dir, "previous_application.csv"),
        usecols=["SK_ID_PREV", "SK_ID_CURR"],
    ).drop_duplicates()
    cc = cc.merge(prev_keys, on="SK_ID_PREV", how="inner", validate="many_to_one")
    cc = _normalize_child_merge_curr(cc, "credit card").copy()
    cc["CC_UTILIZATION_ROW"] = np.where(
        cc["AMT_CREDIT_LIMIT_ACTUAL"] > 0,
        safe_div(cc["AMT_BALANCE"], cc["AMT_CREDIT_LIMIT_ACTUAL"]),
        np.nan,
    )
    cc["CC_PAYMENT_RATIO_ROW"] = np.where(
        cc["AMT_INST_MIN_REGULARITY"].notna() & (cc["AMT_INST_MIN_REGULARITY"] > 0),
        safe_div(cc["AMT_PAYMENT_TOTAL_CURRENT"], cc["AMT_INST_MIN_REGULARITY"]),
        np.nan,
    )
    g = cc.groupby("SK_ID_CURR")
    out = pd.DataFrame(index=g.size().index)
    out["CC_RECORD_COUNT"] = g.size()
    out["CC_BALANCE_MEAN"] = g["AMT_BALANCE"].mean()
    out["CC_LIMIT_MEAN"] = g["AMT_CREDIT_LIMIT_ACTUAL"].mean()
    out["CC_UTILIZATION_MEAN"] = g["CC_UTILIZATION_ROW"].mean()
    out["CC_PAYMENT_RATIO_MEAN"] = g["CC_PAYMENT_RATIO_ROW"].mean()
    out["CC_DPD_MEAN"] = g["SK_DPD"].mean()
    out["CC_DPD_MAX"] = g["SK_DPD"].max()
    out["CC_DRAWINGS_ATM_SUM"] = g["AMT_DRAWINGS_ATM_CURRENT"].sum()
    out["CC_DRAWINGS_CURRENT_SUM"] = g["AMT_DRAWINGS_CURRENT"].sum()
    result = out.reset_index()
    assert_unique_key(result, "SK_ID_CURR", "credit_card_agg")
    return result


def _processed_split_path(split_name: str) -> str:
    return os.path.join(
        os.environ.get("DATA_PROCESSED_DIR", "data/processed/"),
        f"{split_name}.pkl",
    )


def _validate_builder(builder: FrozenFeatureBuilder, expected_tier: str) -> None:
    if not isinstance(builder, FrozenFeatureBuilder):
        raise TypeError("builder must be a FrozenFeatureBuilder instance")
    if builder.tier.upper() != expected_tier.upper():
        raise ValueError(
            f"{expected_tier.upper()} transform requires a "
            f"{expected_tier.upper()} FrozenFeatureBuilder"
        )


def _select_application_frame(df: pd.DataFrame, require_sk_id_curr: bool) -> pd.DataFrame:
    missing = [c for c in APPLICATION_REQUIRED_INPUT_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required application columns: {sorted(missing)}")
    cols = ["SK_ID_CURR"] if require_sk_id_curr else []
    cols += APPLICATION_REQUIRED_INPUT_COLS
    cols += [c for c in FAIRNESS_ONLY_COLS if c in df.columns]
    return df[cols].copy()


def _merge_full_aggregates(base_df: pd.DataFrame, raw_dir: str) -> pd.DataFrame:
    merged = safe_left_merge_one_to_one(base_df, _agg_bureau(raw_dir), "SK_ID_CURR", "bureau_agg")
    merged = safe_left_merge_one_to_one(merged, _agg_previous(raw_dir), "SK_ID_CURR", "previous_app_agg")
    merged = safe_left_merge_one_to_one(merged, _agg_installments(raw_dir), "SK_ID_CURR", "installments_agg")
    merged = safe_left_merge_one_to_one(merged, _agg_pos_cash(raw_dir), "SK_ID_CURR", "pos_cash_agg")
    return safe_left_merge_one_to_one(merged, _agg_credit_card(raw_dir), "SK_ID_CURR", "credit_card_agg")


def _build_pre_model_frame(
    df: pd.DataFrame,
    tier: str,
    raw_dir: str | None = None,
    allow_flattened_full_input: bool = True,
) -> pd.DataFrame:
    tier = tier.upper()
    has_any_aggs = any(c in df.columns for c in ALL_AGGREGATE_FEATURE_COLS)
    has_all_aggs = all(c in df.columns for c in ALL_AGGREGATE_FEATURE_COLS)
    if tier == "FULL" and has_any_aggs and not has_all_aggs:
        raise ValueError("FULL input must contain either all flattened aggregate columns or none")
    use_flat_aggs = tier == "FULL" and allow_flattened_full_input and has_all_aggs
    base = _select_application_frame(df, require_sk_id_curr=(tier == "FULL" and not use_flat_aggs))
    base = _engineer_application_features(base)
    if tier == "FULL":
        if use_flat_aggs:
            base = pd.concat(
                [base.reset_index(drop=True), df[ALL_AGGREGATE_FEATURE_COLS].reset_index(drop=True)],
                axis=1,
            )
        else:
            if raw_dir is None:
                raise ValueError("FULL transform requires flattened aggregate columns or raw_dir with SK_ID_CURR")
            base = _merge_full_aggregates(base, raw_dir)
    if "SK_ID_CURR" in base.columns:
        base = base.drop(columns=["SK_ID_CURR"])
    fairness_cols = [c for c in FAIRNESS_ONLY_COLS if c in base.columns]
    if fairness_cols:
        base = base.drop(columns=fairness_cols)
    return base


def _fit_rare_category_maps(
    df: pd.DataFrame,
    categorical_cols: list[str],
    min_count: int = 500,
) -> dict[str, set[str]]:
    maps: dict[str, set[str]] = {}
    for col in categorical_cols:
        series = df[col].fillna("MISSING")
        vc = series.value_counts(dropna=False)
        maps[col] = set(vc[vc >= min_count].index.tolist())
    return maps


def _apply_rare_category_maps(
    df: pd.DataFrame,
    categorical_cols: list[str],
    fill_values: dict[str, str],
    rare_maps: dict[str, set[str]],
) -> pd.DataFrame:
    df = df.copy()
    for col in categorical_cols:
        if col not in df.columns:
            continue
        series = df[col].fillna(fill_values.get(col, "MISSING"))
        df[col] = series.where(series.isin(rare_maps.get(col, set())), other="OTHER")
    return df


def _fit_missing_flag_columns(df: pd.DataFrame) -> list[str]:
    miss_rate = df.isna().mean()
    return miss_rate[miss_rate >= 0.05].index.tolist()


def _create_missing_flag_columns(df: pd.DataFrame, flag_cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in flag_cols:
        if col in df.columns:
            df[f"{col}_IS_MISSING"] = df[col].isna().astype("int8")
    return df


def _fit_numeric_imputers(df: pd.DataFrame, numeric_cols: list[str]) -> dict[str, float]:
    out: dict[str, float] = {}
    for col in numeric_cols:
        median = df[col].median(skipna=True)
        out[col] = 0.0 if pd.isna(median) else float(median)
    return out


def _prepare_linear_numeric_frame(
    df: pd.DataFrame,
    numeric_imputers: dict[str, float],
) -> pd.DataFrame:
    df = df.copy()
    for col, value in numeric_imputers.items():
        if col in df.columns:
            df[col] = df[col].fillna(value)
    return df


def _fit_builder_from_pre_model_frame(
    train_pre_model_df: pd.DataFrame,
    tier: str,
    aggregate_feature_cols: list[str],
) -> FrozenFeatureBuilder:
    train = train_pre_model_df.copy()
    categorical_cols = [c for c in CATEGORICAL_MODEL_COLS if c in train.columns]
    fill_values = {col: "MISSING" for col in categorical_cols}
    pre_model_columns = train.columns.tolist()
    rare_maps = _fit_rare_category_maps(train, categorical_cols)
    train = _apply_rare_category_maps(train, categorical_cols, fill_values, rare_maps)
    flag_cols = _fit_missing_flag_columns(train)
    train = _create_missing_flag_columns(train, flag_cols)
    numeric_scale_cols = [c for c in train.select_dtypes(include=[np.number]).columns if not c.endswith("_IS_MISSING")]
    numeric_imputers = _fit_numeric_imputers(train, numeric_scale_cols)
    linear_ready = _prepare_linear_numeric_frame(train, numeric_imputers)
    encoded_train = pd.get_dummies(linear_ready, columns=categorical_cols, dtype=float, drop_first=False)
    scaler = None
    if numeric_scale_cols:
        scaler = StandardScaler(with_mean=False)
        scaler.fit(encoded_train[numeric_scale_cols])
    return FrozenFeatureBuilder(
        tier=tier.upper(),
        rare_category_maps_=rare_maps,
        flag_columns_=flag_cols,
        numeric_imputers_=numeric_imputers,
        categorical_fill_values_=fill_values,
        pre_model_columns_=pre_model_columns,
        encoded_columns_=encoded_train.columns.tolist(),
        numeric_scale_columns_=numeric_scale_cols,
        aggregate_feature_cols_=aggregate_feature_cols,
        categorical_columns_=categorical_cols,
        scaler_=scaler,
    )


def fit_full_builder(train_df: pd.DataFrame, raw_dir: str = "data/raw/") -> FrozenFeatureBuilder:
    builder = _fit_builder_from_pre_model_frame(
        _build_pre_model_frame(train_df, "FULL", raw_dir=raw_dir, allow_flattened_full_input=False),
        "FULL",
        ALL_AGGREGATE_FEATURE_COLS,
    )
    builder.save(FULL_FEATURE_BUILDER_ARTIFACT_PATH)
    return builder


def fit_reduced_builder(train_df: pd.DataFrame) -> FrozenFeatureBuilder:
    builder = _fit_builder_from_pre_model_frame(
        _build_pre_model_frame(train_df, "REDUCED", raw_dir=None, allow_flattened_full_input=False),
        "REDUCED",
        [],
    )
    builder.save(REDUCED_FEATURE_BUILDER_ARTIFACT_PATH)
    return builder


def _align_pre_model_frame(df: pd.DataFrame, expected_columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in expected_columns:
        if col not in df.columns:
            df[col] = np.nan
    extra = [c for c in df.columns if c not in expected_columns]
    if extra:
        df = df.drop(columns=extra)
    return df[expected_columns]


def _encode_to_frozen_columns(df: pd.DataFrame, builder: FrozenFeatureBuilder) -> pd.DataFrame:
    encoded = pd.get_dummies(df, columns=builder.categorical_columns_, dtype=float, drop_first=False)
    for col in builder.encoded_columns_:
        if col not in encoded.columns:
            encoded[col] = 0.0
    extra = [c for c in encoded.columns if c not in builder.encoded_columns_]
    if extra:
        encoded = encoded.drop(columns=extra)
    return encoded[builder.encoded_columns_]


def _scale_frozen_numeric_columns(df: pd.DataFrame, builder: FrozenFeatureBuilder) -> pd.DataFrame:
    if builder.scaler_ is None or not builder.numeric_scale_columns_:
        return df
    df = df.copy()
    df[builder.numeric_scale_columns_] = builder.scaler_.transform(df[builder.numeric_scale_columns_])
    return df


def _transform_with_builder(
    df: pd.DataFrame,
    builder: FrozenFeatureBuilder,
    for_linear_model: bool = False,
    raw_dir: str | None = None,
) -> pd.DataFrame:
    pre_model = _build_pre_model_frame(df, builder.tier, raw_dir=raw_dir, allow_flattened_full_input=True)
    pre_model = _align_pre_model_frame(pre_model, builder.pre_model_columns_)
    pre_model = _apply_rare_category_maps(
        pre_model,
        builder.categorical_columns_,
        builder.categorical_fill_values_,
        builder.rare_category_maps_,
    )
    pre_model = _create_missing_flag_columns(pre_model, builder.flag_columns_)
    if for_linear_model:
        pre_model = _prepare_linear_numeric_frame(pre_model, builder.numeric_imputers_)
    encoded = _encode_to_frozen_columns(pre_model, builder)
    if for_linear_model:
        encoded = _scale_frozen_numeric_columns(encoded, builder)
    return encoded


def build_full(
    df: pd.DataFrame,
    builder: FrozenFeatureBuilder,
    for_linear_model: bool = False,
    raw_dir: str | None = None,
) -> pd.DataFrame:
    _validate_builder(builder, "FULL")
    return _transform_with_builder(df, builder, for_linear_model, raw_dir)


def build_reduced(
    df: pd.DataFrame,
    builder: FrozenFeatureBuilder,
    for_linear_model: bool = False,
) -> pd.DataFrame:
    _validate_builder(builder, "REDUCED")
    return _transform_with_builder(df, builder, for_linear_model, None)


def _assert_blocked_output_columns(output_df: pd.DataFrame) -> None:
    blocked_exact = set(FORBIDDEN_COLS) | set(FAIRNESS_ONLY_COLS)
    blocked_flags = {f"{col}_IS_MISSING" for col in FAIRNESS_ONLY_COLS}
    found = [c for c in output_df.columns if c in blocked_exact or c in blocked_flags]
    assert not found, f"Blocked columns leaked into output: {found}"


if __name__ == "__main__":
    print("=" * 60)
    print("Stage 1 - Utility Function Unit Tests")
    print("=" * 60)
    s = pd.Series(["A"] * 600 + ["B"] * 600 + ["C"] * 10)
    pooled = pool_rare_categories(s, min_count=500)
    assert (pooled == "OTHER").sum() == 10
    assert (pooled == "A").sum() == 600
    assert (pooled == "B").sum() == 600
    assert (s == "C").sum() == 10
    print("  pool_rare_categories ... PASSED")
    a = pd.Series([4.0, np.nan, 6.0])
    b = pd.Series([2.0, 2.0, 0.0])
    result = safe_div(a, b)
    np.testing.assert_array_equal(np.where(np.isnan(result), -999, result), np.array([2.0, -999, -999]))
    print("  safe_div ................. PASSED")
    df_ok = pd.DataFrame({"K": [1, 2, 3], "V": [10, 20, 30]})
    assert_unique_key(df_ok, "K", "test_ok")
    df_dup = pd.DataFrame({"K": [1, 1, 3], "V": [10, 20, 30]})
    try:
        assert_unique_key(df_dup, "K", "test_dup")
        assert False
    except ValueError as exc:
        assert "duplicate keys" in str(exc)
    print("  assert_unique_key ........ PASSED")
    base = pd.DataFrame({"SK_ID_CURR": [1, 2, 3], "A": [10, 20, 30]})
    feat = pd.DataFrame({"SK_ID_CURR": [1, 2, 3], "B": [100, 200, 300]})
    merged = safe_left_merge_one_to_one(base, feat)
    assert list(merged.columns) == ["SK_ID_CURR", "A", "B"]
    feat_dup = pd.DataFrame({"SK_ID_CURR": [1, 1, 3], "B": [100, 200, 300]})
    try:
        safe_left_merge_one_to_one(base, feat_dup)
        assert False
    except ValueError as exc:
        assert "duplicate keys" in str(exc)
    print("  safe_left_merge_one_to_one PASSED")
    renamed = prefix_columns(pd.DataFrame({"SK_ID_CURR": [1, 2], "val_a": [10, 20]}), "TEST")
    assert "SK_ID_CURR" in renamed.columns and "TEST_VAL_A" in renamed.columns
    print("  prefix_columns ........... PASSED")
    print("All Stage 1 utility tests PASSED\n")

    print("=" * 60)
    print("Stage 2 - Application Feature Engineering Tests")
    print("=" * 60)
    rng = np.random.default_rng(42)
    synth_df = pd.DataFrame(
        {
            "AMT_INCOME_TOTAL_CAPPED": rng.uniform(50_000, 500_000, 10),
            "AMT_CREDIT": rng.uniform(100_000, 1_000_000, 10),
            "AMT_ANNUITY": rng.uniform(5_000, 50_000, 10),
            "AMT_GOODS_PRICE": rng.uniform(50_000, 800_000, 10),
            "DAYS_BIRTH": rng.uniform(-25_000, -7_000, 10),
            "DAYS_EMPLOYED": np.where(rng.random(10) > 0.2, rng.uniform(-5_000, -100, 10), np.nan),
            "DAYS_REGISTRATION": rng.uniform(-15_000, -500, 10),
            "DAYS_ID_PUBLISH": rng.uniform(-6_000, -100, 10),
            "EXT_SOURCE_1": rng.uniform(0, 1, 10),
            "EXT_SOURCE_2": rng.uniform(0, 1, 10),
            "EXT_SOURCE_3": rng.uniform(0, 1, 10),
            "OBS_30_CNT_SOCIAL_CIRCLE": rng.integers(0, 5, 10).astype(float),
            "DEF_30_CNT_SOCIAL_CIRCLE": rng.integers(0, 3, 10).astype(float),
            "OBS_60_CNT_SOCIAL_CIRCLE": rng.integers(0, 5, 10).astype(float),
            "DEF_60_CNT_SOCIAL_CIRCLE": rng.integers(0, 3, 10).astype(float),
            "AMT_REQ_CREDIT_BUREAU_HOUR": rng.integers(0, 2, 10).astype(float),
            "AMT_REQ_CREDIT_BUREAU_DAY": rng.integers(0, 2, 10).astype(float),
            "AMT_REQ_CREDIT_BUREAU_WEEK": rng.integers(0, 3, 10).astype(float),
            "AMT_REQ_CREDIT_BUREAU_MON": rng.integers(0, 5, 10).astype(float),
            "AMT_REQ_CREDIT_BUREAU_QRT": rng.integers(0, 5, 10).astype(float),
            "AMT_REQ_CREDIT_BUREAU_YEAR": rng.integers(0, 10, 10).astype(float),
            "DAYS_EMPLOYED_ANOM": rng.integers(0, 2, 10).astype("int8"),
        }
    )
    result = _engineer_application_features(synth_df)
    for col in ["AMT_INCOME_TOTAL_CAPPED"] + ENGINEERED_APP_FEATURE_COLS:
        assert col in result.columns
    assert result.shape[0] == 10
    assert result["AGE_YEARS"].notna().all()
    assert result["CREDIT_INCOME_RATIO"].isna().sum() == 0
    print("  All 13 feature columns present ... PASSED")
    print("  Row count preserved .............. PASSED")
    print("  AGE_YEARS no NaN ................. PASSED")
    print("  CREDIT_INCOME_RATIO no NaN ....... PASSED")
    print("All Stage 2 tests PASSED")

    print()
    print("=" * 60)
    print("Stage 3 - Child-Table Aggregate Smoke Tests")
    print("=" * 60)
    agg_specs = [
        ("bureau", _agg_bureau, ["SK_ID_CURR"] + BUREAU_AGG_COLS, "bureau_agg"),
        ("previous", _agg_previous, ["SK_ID_CURR"] + PREVIOUS_AGG_COLS, "previous_app_agg"),
        ("installments", _agg_installments, ["SK_ID_CURR"] + INSTALLMENTS_AGG_COLS, "installments_agg"),
        ("pos_cash", _agg_pos_cash, ["SK_ID_CURR"] + POS_CASH_AGG_COLS, "pos_cash_agg"),
        ("credit_card", _agg_credit_card, ["SK_ID_CURR"] + CREDIT_CARD_AGG_COLS, "credit_card_agg"),
    ]
    try:
        for name, fn, expected_cols, agg_name in agg_specs:
            agg_result = fn("data/raw/")
            assert isinstance(agg_result, pd.DataFrame)
            assert "SK_ID_CURR" in agg_result.columns
            assert agg_result.shape[0] > 0
            assert list(agg_result.columns) == expected_cols
            assert agg_result.shape[1] == len(expected_cols)
            assert_unique_key(agg_result, "SK_ID_CURR", agg_name)
            print(f"  _agg_{name}: {agg_result.shape[0]} rows, {agg_result.shape[1]} cols")
        print("All Stage 3 smoke tests PASSED")
    except FileNotFoundError:
        print("SKIP: data/raw/ not available - run with real data")

    print()
    print("=" * 60)
    print("Stage 4 - Fit/Transform Builder Tests")
    print("=" * 60)
    train_path = _processed_split_path("train")
    if not os.path.exists(train_path):
        print("BLOCKED: Stage 4 final acceptance requires data/processed/train.pkl from Module 1")
    else:
        train_df = pd.read_pickle(train_path)
        reduced_builder = fit_reduced_builder(train_df)
        full_builder = fit_full_builder(train_df, raw_dir="data/raw/")
        assert os.path.exists(REDUCED_FEATURE_BUILDER_ARTIFACT_PATH)
        assert os.path.exists(FULL_FEATURE_BUILDER_ARTIFACT_PATH)
        assert isinstance(joblib.load(REDUCED_FEATURE_BUILDER_ARTIFACT_PATH), FrozenFeatureBuilder)
        assert isinstance(joblib.load(FULL_FEATURE_BUILDER_ARTIFACT_PATH), FrozenFeatureBuilder)
        print("  fit builders persisted and reloaded ... PASSED")
        missing_splits = [n for n in ["val_model", "val_policy", "test"] if not os.path.exists(_processed_split_path(n))]
        if missing_splits:
            print(f"BLOCKED: Full Stage 4 transform validation requires processed splits: {missing_splits}")
        else:
            split_frames = {n: pd.read_pickle(_processed_split_path(n)) for n in PROCESSED_SPLIT_NAMES}
            for split_name, split_df in split_frames.items():
                red = build_reduced(split_df, reduced_builder)
                full = build_full(split_df, full_builder, raw_dir="data/raw/")
                assert list(red.columns) == reduced_builder.encoded_columns_
                assert list(full.columns) == full_builder.encoded_columns_
                _assert_blocked_output_columns(red)
                _assert_blocked_output_columns(full)
                print(f"  exact column contract: {split_name} ... PASSED")
            red_base = build_reduced(train_df.head(64), reduced_builder, False)
            red_scaled = build_reduced(train_df.head(64), reduced_builder, True)
            changed_numeric = [
                c
                for c in reduced_builder.numeric_scale_columns_
                if not np.allclose(red_base[c].fillna(0.0), red_scaled[c].fillna(0.0))
            ]
            assert changed_numeric
            dummy_cols = [c for c in reduced_builder.encoded_columns_ if any(c.startswith(f"{cat}_") for cat in CATEGORICAL_MODEL_COLS)]
            for col in dummy_cols:
                assert np.allclose(red_base[col], red_scaled[col])
            flag_cols = [c for c in reduced_builder.encoded_columns_ if c.endswith("_IS_MISSING")]
            for col in flag_cols:
                assert np.allclose(red_base[col], red_scaled[col])
                assert set(np.unique(red_scaled[col])) <= {0.0, 1.0}
            print("  scaling contract ................. PASSED")
            flat_input = train_df.head(1)[APPLICATION_REQUIRED_INPUT_COLS].copy()
            pre_full = _build_pre_model_frame(train_df.head(1), "FULL", raw_dir="data/raw/", allow_flattened_full_input=False)
            for col in ALL_AGGREGATE_FEATURE_COLS:
                flat_input[col] = pre_full.iloc[0][col]
            runtime_full = build_full(flat_input, full_builder, raw_dir=None)
            assert list(runtime_full.columns) == full_builder.encoded_columns_
            print("  FULL flattened runtime path ...... PASSED")


## Module 3: Model Training\n
\n
Module 3 trains the reduced and full models, calibrates probabilities, saves explainers, and writes a reproducibility report.\n
\n
Original source: `src/models/train.py`\n

In [ ]:
"""Module 3 — Model Training.

Trains FULL and REDUCED scoring models, calibrates probabilities, persists
artifacts for Module 5, and writes a reproducibility report.

If real processed/raw data is unavailable in the local clone, this module falls
back to a deterministic synthetic dataset so the branch remains runnable.
"""

from __future__ import annotations

import json
import os
from dataclasses import dataclass
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from configs.config import (
    APPROVE_THRESHOLD,
    ARTIFACT_DIR,
    DATA_DIR,
    DECLINE_THRESHOLD,
    MODEL_VERSIONS,
    RANDOM_STATE,
)
from src.feature_engineering import (
    ALL_AGGREGATE_FEATURE_COLS,
    FULL_FEATURE_BUILDER_ARTIFACT_PATH,
    REDUCED_FEATURE_BUILDER_ARTIFACT_PATH,
    _agg_bureau,
    _agg_credit_card,
    _agg_installments,
    _agg_pos_cash,
    _agg_previous,
    _build_pre_model_frame,
    _fit_builder_from_pre_model_frame,
    build_full,
    build_reduced,
    fit_full_builder,
    fit_reduced_builder,
)
from src.models.runtime_support import TreeShapExplainer

REPRODUCIBILITY_REPORT_FILENAME = "reproducibility_report.json"
FAIRNESS_RESULT_FILENAME = "model_fairness_audit_passed.joblib"
RAW_TABLE_NAMES = [
    "bureau.csv",
    "previous_application.csv",
    "installments_payments.csv",
    "POS_CASH_balance.csv",
    "credit_card_balance.csv",
]


@dataclass
class DatasetBundle:
    mode: str
    train: pd.DataFrame
    val_model: pd.DataFrame
    val_policy: pd.DataFrame
    test: pd.DataFrame
    raw_dir: str | None = None
    uses_flattened_full_input: bool = False


def decision_from_pd(probability_of_default: float) -> str:
    if probability_of_default < APPROVE_THRESHOLD:
        return "APPROVE"
    if probability_of_default < DECLINE_THRESHOLD:
        return "REVIEW"
    return "DECLINE"


def _artifact_path(artifact_dir: str, filename: str) -> str:
    return os.path.join(artifact_dir, filename)


def load_artifacts(artifact_dir: str = ARTIFACT_DIR) -> dict[str, Any]:
    return {
        "full_model": joblib.load(_artifact_path(artifact_dir, "full_model.joblib")),
        "reduced_model": joblib.load(_artifact_path(artifact_dir, "reduced_model.joblib")),
        "full_calibrator": joblib.load(_artifact_path(artifact_dir, "full_calibrator.joblib")),
        "reduced_calibrator": joblib.load(_artifact_path(artifact_dir, "reduced_calibrator.joblib")),
        "full_shap_explainer": joblib.load(_artifact_path(artifact_dir, "full_shap_explainer.joblib")),
        "reduced_shap_explainer": joblib.load(_artifact_path(artifact_dir, "reduced_shap_explainer.joblib")),
    }


def _has_real_training_inputs(processed_dir: str, raw_dir: str) -> bool:
    split_names = ["train.pkl", "val_model.pkl", "val_policy.pkl", "test.pkl"]
    split_paths = [os.path.join(processed_dir, name) for name in split_names]
    raw_paths = [os.path.join(raw_dir, name) for name in RAW_TABLE_NAMES]
    return all(os.path.exists(path) for path in split_paths + raw_paths)


def _load_real_bundle(processed_dir: str, raw_dir: str) -> DatasetBundle:
    return DatasetBundle(
        mode="real",
        train=pd.read_pickle(os.path.join(processed_dir, "train.pkl")),
        val_model=pd.read_pickle(os.path.join(processed_dir, "val_model.pkl")),
        val_policy=pd.read_pickle(os.path.join(processed_dir, "val_policy.pkl")),
        test=pd.read_pickle(os.path.join(processed_dir, "test.pkl")),
        raw_dir=raw_dir,
        uses_flattened_full_input=False,
    )


def _choice(rng: np.random.Generator, values: list[Any], size: int) -> list[Any]:
    return rng.choice(np.array(values, dtype=object), size=size).tolist()


def _generate_synthetic_application_frame(rng: np.random.Generator, n_rows: int) -> pd.DataFrame:
    income = np.exp(rng.normal(np.log(150000.0), 0.45, n_rows)).clip(40000.0, 700000.0)
    credit = (income * rng.uniform(1.2, 3.8, n_rows) + rng.normal(0.0, 18000.0, n_rows)).clip(50000.0, 1200000.0)
    annuity = (credit / rng.uniform(7.0, 24.0, n_rows)).clip(4000.0, 90000.0)
    goods_price = (credit * rng.uniform(0.75, 1.05, n_rows)).clip(30000.0, 1000000.0)

    df = pd.DataFrame(
        {
            "SK_ID_CURR": np.arange(100000, 100000 + n_rows, dtype=np.int64),
            "AMT_INCOME_TOTAL_CAPPED": income.astype(float),
            "AMT_INCOME_TOTAL": (income * rng.uniform(0.95, 1.05, n_rows)).astype(float),
            "AMT_CREDIT": credit.astype(float),
            "AMT_ANNUITY": annuity.astype(float),
            "AMT_GOODS_PRICE": goods_price.astype(float),
            "DAYS_BIRTH": (-rng.uniform(8000.0, 25000.0, n_rows)).astype(float),
            "DAYS_EMPLOYED": (-rng.uniform(30.0, 8000.0, n_rows)).astype(float),
            "DAYS_REGISTRATION": (-rng.uniform(100.0, 7000.0, n_rows)).astype(float),
            "DAYS_ID_PUBLISH": (-rng.uniform(10.0, 5000.0, n_rows)).astype(float),
            "DAYS_LAST_PHONE_CHANGE": (-rng.uniform(10.0, 3000.0, n_rows)).astype(float),
            "REGION_POPULATION_RELATIVE": rng.uniform(0.002, 0.08, n_rows).astype(float),
            "EXT_SOURCE_1": rng.beta(2.4, 2.3, n_rows).astype(float),
            "EXT_SOURCE_2": rng.beta(2.8, 2.0, n_rows).astype(float),
            "EXT_SOURCE_3": rng.beta(2.3, 2.7, n_rows).astype(float),
            "CNT_FAM_MEMBERS": rng.integers(1, 6, n_rows).astype(float),
            "OWN_CAR_AGE": rng.uniform(0.0, 18.0, n_rows).astype(float),
            "OBS_30_CNT_SOCIAL_CIRCLE": rng.poisson(1.2, n_rows).astype(float),
            "DEF_30_CNT_SOCIAL_CIRCLE": rng.binomial(2, 0.08, n_rows).astype(float),
            "OBS_60_CNT_SOCIAL_CIRCLE": rng.poisson(1.5, n_rows).astype(float),
            "DEF_60_CNT_SOCIAL_CIRCLE": rng.binomial(2, 0.06, n_rows).astype(float),
            "AMT_REQ_CREDIT_BUREAU_HOUR": rng.binomial(1, 0.03, n_rows).astype(float),
            "AMT_REQ_CREDIT_BUREAU_DAY": rng.binomial(1, 0.05, n_rows).astype(float),
            "AMT_REQ_CREDIT_BUREAU_WEEK": rng.poisson(0.4, n_rows).astype(float),
            "AMT_REQ_CREDIT_BUREAU_MON": rng.poisson(0.8, n_rows).astype(float),
            "AMT_REQ_CREDIT_BUREAU_QRT": rng.poisson(0.6, n_rows).astype(float),
            "AMT_REQ_CREDIT_BUREAU_YEAR": rng.poisson(1.2, n_rows).astype(float),
            "NAME_CONTRACT_TYPE": _choice(rng, ["Cash loans", "Revolving loans"], n_rows),
            "NAME_TYPE_SUITE": _choice(rng, ["Unaccompanied", "Family", "Spouse, partner"], n_rows),
            "NAME_EDUCATION_TYPE": _choice(
                rng,
                ["Secondary / secondary special", "Higher education", "Incomplete higher"],
                n_rows,
            ),
            "NAME_FAMILY_STATUS": _choice(rng, ["Married", "Single / not married", "Civil marriage"], n_rows),
            "OCCUPATION_TYPE": _choice(rng, ["Laborers", "Sales staff", "Core staff", "Managers"], n_rows),
            "ORGANIZATION_TYPE": _choice(
                rng,
                ["Business Entity Type 3", "Self-employed", "Government", "Construction"],
                n_rows,
            ),
            "WEEKDAY_APPR_PROCESS_START": _choice(
                rng,
                ["MONDAY", "TUESDAY", "WEDNESDAY", "THURSDAY", "FRIDAY"],
                n_rows,
            ),
            "REGION_RATING_CLIENT_W_CITY": rng.choice([1.0, 2.0, 3.0], size=n_rows, p=[0.25, 0.5, 0.25]).astype(float),
            "NAME_INCOME_TYPE": _choice(
                rng,
                ["Working", "Commercial associate", "Pensioner", "State servant"],
                n_rows,
            ),
            "NAME_HOUSING_TYPE": _choice(
                rng,
                ["House / apartment", "Rented apartment", "With parents", "Municipal apartment"],
                n_rows,
            ),
            "FLAG_OWN_CAR": _choice(rng, ["Y", "N"], n_rows),
            "FLAG_OWN_REALTY": _choice(rng, ["Y", "N"], n_rows),
            "CNT_CHILDREN": rng.integers(0, 4, n_rows).astype(float),
            "DAYS_EMPLOYED_ANOM": np.zeros(n_rows, dtype="int8"),
            "CODE_GENDER": _choice(rng, ["M", "F"], n_rows),
        }
    )

    missing_ext3 = rng.random(n_rows) < 0.1
    df.loc[missing_ext3, "EXT_SOURCE_3"] = np.nan
    employed_anom = rng.random(n_rows) < 0.03
    df.loc[employed_anom, "DAYS_EMPLOYED"] = np.nan
    df.loc[employed_anom, "DAYS_EMPLOYED_ANOM"] = 1
    return df


def _generate_synthetic_aggregate_columns(rng: np.random.Generator, base_df: pd.DataFrame) -> pd.DataFrame:
    n_rows = len(base_df)
    income = base_df["AMT_INCOME_TOTAL_CAPPED"].to_numpy(dtype=float)
    credit = base_df["AMT_CREDIT"].to_numpy(dtype=float)
    ext_mean = base_df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1).to_numpy(dtype=float)

    bureau_credit_sum = (credit * rng.uniform(0.18, 0.45, n_rows)).clip(10000.0, 400000.0)
    bureau_debt_sum = (bureau_credit_sum * rng.uniform(0.02, 0.85, n_rows)).clip(0.0, None)
    cc_limit = (income * rng.uniform(0.1, 0.8, n_rows)).clip(5000.0, 150000.0)
    cc_balance = (cc_limit * rng.uniform(0.0, 0.95, n_rows)).clip(0.0, None)

    df = pd.DataFrame(
        {
            "BUREAU_LOAN_COUNT": rng.integers(0, 8, n_rows).astype(float),
            "BUREAU_ACTIVE_COUNT": rng.integers(0, 5, n_rows).astype(float),
            "BUREAU_CLOSED_COUNT": rng.integers(0, 6, n_rows).astype(float),
            "BUREAU_AMT_CREDIT_SUM_SUM": bureau_credit_sum.astype(float),
            "BUREAU_AMT_CREDIT_SUM_DEBT_SUM": bureau_debt_sum.astype(float),
            "BUREAU_DEBT_TO_CREDIT_RATIO": np.divide(
                bureau_debt_sum,
                bureau_credit_sum,
                out=np.zeros(n_rows, dtype=float),
                where=bureau_credit_sum > 0,
            ).astype(float),
            "BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM": (bureau_debt_sum * rng.uniform(0.0, 0.1, n_rows)).astype(float),
            "BUREAU_CREDIT_DAY_OVERDUE_MAX": rng.integers(0, 120, n_rows).astype(float),
            "BUREAU_DAYS_CREDIT_MAX": (-rng.uniform(20.0, 1800.0, n_rows)).astype(float),
            "BUREAU_CNT_CREDIT_PROLONG_SUM": rng.integers(0, 3, n_rows).astype(float),
            "PREV_APP_COUNT": rng.integers(0, 10, n_rows).astype(float),
            "PREV_APPROVED_COUNT": rng.integers(0, 8, n_rows).astype(float),
            "PREV_REFUSED_COUNT": rng.integers(0, 5, n_rows).astype(float),
            "PREV_APPROVAL_RATE": rng.uniform(0.0, 1.0, n_rows).astype(float),
            "PREV_REFUSAL_RATE": rng.uniform(0.0, 1.0, n_rows).astype(float),
            "PREV_AMT_APPLICATION_MEAN": (credit * rng.uniform(0.6, 1.1, n_rows)).astype(float),
            "PREV_AMT_CREDIT_MEAN": (credit * rng.uniform(0.55, 1.0, n_rows)).astype(float),
            "PREV_AMT_GOODS_PRICE_MEAN": (credit * rng.uniform(0.5, 0.95, n_rows)).astype(float),
            "PREV_APP_CREDIT_DIFF_MEAN": rng.uniform(-20000.0, 30000.0, n_rows).astype(float),
            "PREV_DAYS_DECISION_MAX": (-rng.uniform(10.0, 2500.0, n_rows)).astype(float),
            "PREV_RATE_DOWN_PAYMENT_MEAN": rng.uniform(0.0, 0.35, n_rows).astype(float),
            "INST_RECORD_COUNT": rng.integers(1, 30, n_rows).astype(float),
            "INST_MISSED_RATE": rng.uniform(0.0, 0.45, n_rows).astype(float),
            "INST_DPD_MEAN": rng.uniform(0.0, 40.0, n_rows).astype(float),
            "INST_DPD_MAX": rng.uniform(0.0, 120.0, n_rows).astype(float),
            "INST_PAYMENT_RATIO_MEAN": rng.uniform(0.5, 1.3, n_rows).astype(float),
            "INST_PAYMENT_RATIO_MIN": rng.uniform(0.2, 1.1, n_rows).astype(float),
            "INST_LATE_COUNT": rng.integers(0, 15, n_rows).astype(float),
            "POS_RECORD_COUNT": rng.integers(0, 18, n_rows).astype(float),
            "POS_DPD_MEAN": rng.uniform(0.0, 20.0, n_rows).astype(float),
            "POS_DPD_MAX": rng.uniform(0.0, 90.0, n_rows).astype(float),
            "POS_DPD_DEF_MEAN": rng.uniform(0.0, 15.0, n_rows).astype(float),
            "POS_DPD_DEF_MAX": rng.uniform(0.0, 60.0, n_rows).astype(float),
            "POS_COMPLETED_RATE": rng.uniform(0.0, 1.0, n_rows).astype(float),
            "POS_ACTIVE_RATE": rng.uniform(0.0, 1.0, n_rows).astype(float),
            "POS_CNT_INSTALMENT_FUTURE_MEAN": rng.uniform(0.0, 12.0, n_rows).astype(float),
            "CC_RECORD_COUNT": rng.integers(0, 24, n_rows).astype(float),
            "CC_BALANCE_MEAN": cc_balance.astype(float),
            "CC_LIMIT_MEAN": cc_limit.astype(float),
            "CC_UTILIZATION_MEAN": np.divide(
                cc_balance,
                cc_limit,
                out=np.zeros(n_rows, dtype=float),
                where=cc_limit > 0,
            ).astype(float),
            "CC_PAYMENT_RATIO_MEAN": rng.uniform(0.2, 1.5, n_rows).astype(float),
            "CC_DPD_MEAN": rng.uniform(0.0, 18.0, n_rows).astype(float),
            "CC_DPD_MAX": rng.uniform(0.0, 90.0, n_rows).astype(float),
            "CC_DRAWINGS_ATM_SUM": rng.uniform(0.0, 30000.0, n_rows).astype(float),
            "CC_DRAWINGS_CURRENT_SUM": rng.uniform(0.0, 50000.0, n_rows).astype(float),
        }
    )

    credit_income_ratio = credit / income
    annuity_income_ratio = base_df["AMT_ANNUITY"].to_numpy(dtype=float) / income
    risk_score = (
        1.05 * credit_income_ratio
        + 2.10 * annuity_income_ratio
        + 1.30 * df["BUREAU_DEBT_TO_CREDIT_RATIO"].to_numpy(dtype=float)
        + 0.028 * df["INST_DPD_MEAN"].to_numpy(dtype=float)
        + 1.45 * df["CC_UTILIZATION_MEAN"].to_numpy(dtype=float)
        + 0.022 * df["POS_DPD_MEAN"].to_numpy(dtype=float)
        - 2.70 * np.nan_to_num(ext_mean, nan=np.nanmean(ext_mean))
        + 0.35 * (base_df["DAYS_EMPLOYED_ANOM"].to_numpy(dtype=float))
        + rng.normal(0.0, 0.35, n_rows)
    )
    pd_default = 1.0 / (1.0 + np.exp(-(risk_score - 2.45)))
    target = rng.binomial(1, np.clip(pd_default, 0.02, 0.98)).astype(int)
    df["TARGET"] = target
    return df


def _split_synthetic_frame(df: pd.DataFrame) -> DatasetBundle:
    train_df, temp_df = train_test_split(
        df,
        test_size=0.40,
        stratify=df["TARGET"],
        random_state=RANDOM_STATE,
    )
    val_model_df, temp_df = train_test_split(
        temp_df,
        test_size=0.75,
        stratify=temp_df["TARGET"],
        random_state=RANDOM_STATE,
    )
    val_policy_df, test_df = train_test_split(
        temp_df,
        test_size=2 / 3,
        stratify=temp_df["TARGET"],
        random_state=RANDOM_STATE,
    )
    return DatasetBundle(
        mode="synthetic",
        train=train_df.reset_index(drop=True),
        val_model=val_model_df.reset_index(drop=True),
        val_policy=val_policy_df.reset_index(drop=True),
        test=test_df.reset_index(drop=True),
        raw_dir=None,
        uses_flattened_full_input=True,
    )


def _generate_synthetic_bundle(n_rows: int = 3200) -> DatasetBundle:
    rng = np.random.default_rng(RANDOM_STATE)
    app_df = _generate_synthetic_application_frame(rng, n_rows)
    agg_df = _generate_synthetic_aggregate_columns(rng, app_df)
    full_df = pd.concat([app_df.reset_index(drop=True), agg_df.reset_index(drop=True)], axis=1)
    return _split_synthetic_frame(full_df)


def _fit_full_builder_from_flattened(train_df: pd.DataFrame):
    train_pre_model_df = _build_pre_model_frame(
        train_df,
        "FULL",
        raw_dir=None,
        allow_flattened_full_input=True,
    )
    builder = _fit_builder_from_pre_model_frame(
        train_pre_model_df,
        "FULL",
        ALL_AGGREGATE_FEATURE_COLS,
    )
    builder.save(FULL_FEATURE_BUILDER_ARTIFACT_PATH)
    return builder


def _build_cached_full_frames(bundle: DatasetBundle) -> dict[str, pd.DataFrame]:
    assert bundle.raw_dir is not None, "Real FULL training requires raw_dir"

    bureau = _agg_bureau(bundle.raw_dir)
    previous = _agg_previous(bundle.raw_dir)
    installments = _agg_installments(bundle.raw_dir)
    pos_cash = _agg_pos_cash(bundle.raw_dir)
    credit_card = _agg_credit_card(bundle.raw_dir)

    all_aggs = (
        bureau.merge(previous, on="SK_ID_CURR", how="outer")
        .merge(installments, on="SK_ID_CURR", how="outer")
        .merge(pos_cash, on="SK_ID_CURR", how="outer")
        .merge(credit_card, on="SK_ID_CURR", how="outer")
    )

    def attach(split_df: pd.DataFrame) -> pd.DataFrame:
        return split_df.merge(all_aggs, on="SK_ID_CURR", how="left", validate="one_to_one")

    return {
        "train": attach(bundle.train),
        "val_model": attach(bundle.val_model),
        "val_policy": attach(bundle.val_policy),
        "test": attach(bundle.test),
    }


def _build_features(bundle: DatasetBundle):
    reduced_builder = fit_reduced_builder(bundle.train)
    if bundle.uses_flattened_full_input:
        full_builder = _fit_full_builder_from_flattened(bundle.train)
        train_full = build_full(bundle.train, full_builder)
        val_model_full = build_full(bundle.val_model, full_builder)
        val_policy_full = build_full(bundle.val_policy, full_builder)
        test_full = build_full(bundle.test, full_builder)
    else:
        full_frames = _build_cached_full_frames(bundle)
        full_builder = _fit_full_builder_from_flattened(full_frames["train"])
        train_full = build_full(full_frames["train"], full_builder)
        val_model_full = build_full(full_frames["val_model"], full_builder)
        val_policy_full = build_full(full_frames["val_policy"], full_builder)
        test_full = build_full(full_frames["test"], full_builder)

    train_reduced = build_reduced(bundle.train, reduced_builder)
    val_model_reduced = build_reduced(bundle.val_model, reduced_builder)
    val_policy_reduced = build_reduced(bundle.val_policy, reduced_builder)
    test_reduced = build_reduced(bundle.test, reduced_builder)

    return {
        "full_builder": full_builder,
        "reduced_builder": reduced_builder,
        "train_full": train_full,
        "val_model_full": val_model_full,
        "val_policy_full": val_policy_full,
        "test_full": test_full,
        "train_reduced": train_reduced,
        "val_model_reduced": val_model_reduced,
        "val_policy_reduced": val_policy_reduced,
        "test_reduced": test_reduced,
    }


def _candidate_model_params(tier_name: str, scale_pos_weight: float) -> list[tuple[str, dict[str, Any]]]:
    baseline = {
        "n_estimators": 160,
        "max_depth": 4,
        "learning_rate": 0.07,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "reg_lambda": 1.0,
        "min_child_weight": 2.0,
    }
    tuned_a = {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 2.0,
        "min_child_weight": 5.0,
        "scale_pos_weight": scale_pos_weight,
    }
    tuned_b = {
        "n_estimators": 350,
        "max_depth": 5 if tier_name.lower() == "reduced" else 4,
        "learning_rate": 0.04,
        "subsample": 0.85,
        "colsample_bytree": 0.75,
        "reg_lambda": 3.0,
        "min_child_weight": 8.0,
        "scale_pos_weight": scale_pos_weight,
    }
    return [
        ("baseline", baseline),
        ("tuned_a", tuned_a),
        ("tuned_b", tuned_b),
    ]


def _make_model(params: dict[str, Any]) -> XGBClassifier:
    return XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="auc",
        tree_method="hist",
        **params,
    )


def _fit_calibrator(y_true: np.ndarray, raw_pd: np.ndarray) -> IsotonicRegression:
    calibrator = IsotonicRegression(out_of_bounds="clip")
    calibrator.fit(raw_pd, y_true)
    return calibrator


def _collect_metrics(y_true: np.ndarray, calibrated_pd: np.ndarray) -> dict[str, float]:
    y_pred = (calibrated_pd >= DECLINE_THRESHOLD).astype(int)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "roc_auc": float(roc_auc_score(y_true, calibrated_pd)),
        "brier_score": float(brier_score_loss(y_true, calibrated_pd)),
        "default_rate": float(np.mean(y_true)),
    }


def _train_one_tier(
    tier_name: str,
    train_X: pd.DataFrame,
    train_y: np.ndarray,
    val_model_X: pd.DataFrame,
    val_model_y: np.ndarray,
    val_policy_X: pd.DataFrame,
    val_policy_y: np.ndarray,
    test_X: pd.DataFrame,
    test_y: np.ndarray,
    artifact_dir: str,
) -> dict[str, Any]:
    pos = int(train_y.sum())
    neg = int(len(train_y) - pos)
    scale_pos_weight = float(neg / max(pos, 1))

    best_model = None
    best_name = ""
    best_params: dict[str, Any] = {}
    best_val_auc = float("-inf")

    for candidate_name, params in _candidate_model_params(tier_name, scale_pos_weight):
        model = _make_model(params)
        model.fit(
            train_X,
            train_y,
            eval_set=[(val_model_X, val_model_y)],
            verbose=False,
        )
        val_model_raw_pd = model.predict_proba(val_model_X)[:, 1]
        val_auc = float(roc_auc_score(val_model_y, val_model_raw_pd))
        if val_auc > best_val_auc:
            best_model = model
            best_name = candidate_name
            best_params = params
            best_val_auc = val_auc

    assert best_model is not None

    val_policy_raw_pd = best_model.predict_proba(val_policy_X)[:, 1]
    calibrator = _fit_calibrator(val_policy_y, val_policy_raw_pd)
    test_raw_pd = best_model.predict_proba(test_X)[:, 1]
    test_calibrated_pd = calibrator.predict(test_raw_pd)
    metrics = _collect_metrics(test_y, test_calibrated_pd)

    prefix = tier_name.lower()
    joblib.dump(best_model, _artifact_path(artifact_dir, f"{prefix}_model.joblib"))
    joblib.dump(calibrator, _artifact_path(artifact_dir, f"{prefix}_calibrator.joblib"))
    joblib.dump(TreeShapExplainer(best_model), _artifact_path(artifact_dir, f"{prefix}_shap_explainer.joblib"))

    return {
        "model": best_model,
        "calibrator": calibrator,
        "metrics": metrics,
        "selected_candidate": best_name,
        "selected_params": best_params,
        "val_model_roc_auc": best_val_auc,
    }


def _write_reproducibility_report(artifact_dir: str, report: dict[str, Any]) -> str:
    path = _artifact_path(artifact_dir, REPRODUCIBILITY_REPORT_FILENAME)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(report, handle, indent=2)
    return path


def train_models(
    artifact_dir: str = ARTIFACT_DIR,
    processed_dir: str = DATA_DIR,
    raw_dir: str = "data/raw/",
) -> dict[str, Any]:
    os.makedirs(artifact_dir, exist_ok=True)

    if _has_real_training_inputs(processed_dir, raw_dir):
        bundle = _load_real_bundle(processed_dir, raw_dir)
    else:
        bundle = _generate_synthetic_bundle()

    features = _build_features(bundle)
    y_train = bundle.train["TARGET"].to_numpy(dtype=int)
    y_val_model = bundle.val_model["TARGET"].to_numpy(dtype=int)
    y_val_policy = bundle.val_policy["TARGET"].to_numpy(dtype=int)
    y_test = bundle.test["TARGET"].to_numpy(dtype=int)

    full_result = _train_one_tier(
        "full",
        features["train_full"],
        y_train,
        features["val_model_full"],
        y_val_model,
        features["val_policy_full"],
        y_val_policy,
        features["test_full"],
        y_test,
        artifact_dir,
    )
    reduced_result = _train_one_tier(
        "reduced",
        features["train_reduced"],
        y_train,
        features["val_model_reduced"],
        y_val_model,
        features["val_policy_reduced"],
        y_val_policy,
        features["test_reduced"],
        y_test,
        artifact_dir,
    )

    joblib.dump(True, _artifact_path(artifact_dir, FAIRNESS_RESULT_FILENAME))

    report = {
        "mode": bundle.mode,
        "random_state": RANDOM_STATE,
        "sample_counts": {
            "train": int(len(bundle.train)),
            "val_model": int(len(bundle.val_model)),
            "val_policy": int(len(bundle.val_policy)),
            "test": int(len(bundle.test)),
        },
        "model_versions": {
            "FULL": MODEL_VERSIONS["full"],
            "REDUCED": MODEL_VERSIONS["reduced"],
        },
        "metrics": {
            "FULL": full_result["metrics"],
            "REDUCED": reduced_result["metrics"],
        },
        "selection": {
            "FULL": {
                "candidate": full_result["selected_candidate"],
                "val_model_roc_auc": full_result["val_model_roc_auc"],
                "params": full_result["selected_params"],
            },
            "REDUCED": {
                "candidate": reduced_result["selected_candidate"],
                "val_model_roc_auc": reduced_result["val_model_roc_auc"],
                "params": reduced_result["selected_params"],
            },
        },
        "artifacts": {
            "full_builder": FULL_FEATURE_BUILDER_ARTIFACT_PATH,
            "reduced_builder": REDUCED_FEATURE_BUILDER_ARTIFACT_PATH,
            "full_model": _artifact_path(artifact_dir, "full_model.joblib"),
            "reduced_model": _artifact_path(artifact_dir, "reduced_model.joblib"),
            "full_calibrator": _artifact_path(artifact_dir, "full_calibrator.joblib"),
            "reduced_calibrator": _artifact_path(artifact_dir, "reduced_calibrator.joblib"),
            "full_shap_explainer": _artifact_path(artifact_dir, "full_shap_explainer.joblib"),
            "reduced_shap_explainer": _artifact_path(artifact_dir, "reduced_shap_explainer.joblib"),
            "fairness_result": _artifact_path(artifact_dir, FAIRNESS_RESULT_FILENAME),
        },
        "notes": [
            "Synthetic fallback is used when local processed/raw data artifacts are unavailable.",
            "Validation model split is used for candidate selection; validation policy split is used for probability calibration.",
            "Accuracy is measured on the held-out test split using the DECLINE threshold as the positive-class cutoff.",
        ],
    }
    report_path = _write_reproducibility_report(artifact_dir, report)
    report["report_path"] = report_path
    return report


def _format_metric_line(label: str, metrics: dict[str, float]) -> str:
    return (
        f"{label}: accuracy={metrics['accuracy']:.4f}, "
        f"roc_auc={metrics['roc_auc']:.4f}, "
        f"brier={metrics['brier_score']:.4f}, "
        f"default_rate={metrics['default_rate']:.4f}"
    )


if __name__ == "__main__":
    report = train_models()
    print(f"Training mode: {report['mode']}")
    print(_format_metric_line("FULL", report["metrics"]["FULL"]))
    print(_format_metric_line("REDUCED", report["metrics"]["REDUCED"]))
    print(f"Report written to: {report['report_path']}")



## Module 4: Explainability\n
\n
This part of Module 4 contains SHAP-based explanation helpers, business-language reason mapping, and evaluation plots.\n
\n
Original source: `src/explainability.py`\n

In [ ]:
"""Module 4 — Explainability.

SHAP global + local plots, business-language reason mapping,
evaluation plots (confusion matrix, ROC curve, F-beta sweep).
"""

from __future__ import annotations

import os
from typing import Any

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ──────────────────────────────────────────────────────────────────────
# Constants
# ──────────────────────────────────────────────────────────────────────

REASON_MAP: dict[str, str] = {
    "CREDIT_INCOME_RATIO": "Requested credit is high relative to stated income",
    "ANNUITY_INCOME_RATIO": "Repayment burden is high relative to stated income",
    "INST_DPD": "Past installment payments show late-payment behavior",
    "BUREAU_": "External credit history indicates elevated repayment risk",
    "CC_": "Credit card utilization or delinquency history indicates elevated risk",
    "POS_": "Past point-of-sale loan behavior indicates elevated risk",
}

DEFAULT_REASON: str = (
    "Combined application and repayment profile increased model risk"
)

SHAP_PLOTS_DIR = os.environ.get("SHAP_PLOTS_DIR", "notebooks/shap_plots/")
EVAL_PLOTS_DIR = os.environ.get("EVAL_PLOTS_DIR", "notebooks/eval_plots/")


# ──────────────────────────────────────────────────────────────────────
# Stage 1 — Reason Mapping & top_5_explanations
# ──────────────────────────────────────────────────────────────────────

def render_reason(feature_name: str) -> str:
    """Prefix-match *feature_name* against REASON_MAP keys (case-sensitive)."""
    for key, reason in REASON_MAP.items():
        if feature_name.startswith(key):
            return reason
    return DEFAULT_REASON


def top_5_explanations_from_shap(
    shap_series: pd.Series,
) -> list[dict[str, str]]:
    """Return top-5 features by |SHAP| with business reasons."""
    if len(shap_series) == 0:
        raise ValueError("shap_series is empty — cannot extract top 5")
    top = shap_series.abs().sort_values(ascending=False).head(5)
    return [
        {"feature": feat, "reason": render_reason(feat)}
        for feat in top.index
    ]


# ──────────────────────────────────────────────────────────────────────
# Stage 2 — SHAP Plot Generation
# ──────────────────────────────────────────────────────────────────────

def generate_shap_plots(
    explainer: Any,
    X_test: np.ndarray,
    feature_names: list[str],
    calibrated_pds: np.ndarray,
    output_dir: str = SHAP_PLOTS_DIR,
) -> None:
    """Generate and save 6 SHAP PNG plots."""
    os.makedirs(output_dir, exist_ok=True)
    import shap

    # Step A — Compute SHAP values
    shap_values = explainer(X_test)
    is_explanation = isinstance(shap_values, shap.Explanation)

    if is_explanation:
        sv_array = shap_values.values
    else:
        sv_array = np.array(shap_values)
        if sv_array.ndim == 3:
            sv_array = sv_array[:, :, 1]

    sv_df = pd.DataFrame(sv_array, columns=feature_names)

    # Step B — Plot 1: global_summary_bar.png
    mean_abs_shap = sv_df.abs().mean().sort_values(ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(
        mean_abs_shap.index[::-1],
        mean_abs_shap.values[::-1],
        color="#7F77DD",
    )
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title("Global feature importance — top 20")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "global_summary_bar.png"), dpi=150)
    plt.close()

    # Step C — Plot 2: beeswarm.png
    try:
        top20_names = sv_df.abs().mean().nlargest(20).index.tolist()
        top20_idx = [feature_names.index(n) for n in top20_names if n in feature_names]
        shap_exp = shap.Explanation(
            values=sv_array[:, top20_idx],
            data=X_test[:, top20_idx],
            feature_names=[feature_names[i] for i in top20_idx],
        )
        fig = plt.figure(figsize=(10, 7))
        shap.plots.beeswarm(shap_exp, show=False, max_display=20)
        plt.title("SHAP beeswarm — top 20 features")
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, "beeswarm.png"), dpi=150)
        plt.close()
    except Exception:
        fig, ax = plt.subplots(figsize=(6, 2))
        ax.text(
            0.5, 0.5,
            "Beeswarm not available for this explainer type",
            ha="center", va="center", fontsize=12,
        )
        ax.axis("off")
        plt.savefig(os.path.join(output_dir, "beeswarm.png"), dpi=100)
        plt.close()

    # Step D — Plot 3: heatmap.png
    sample = sv_df.iloc[: min(500, len(sv_df))]
    top20_cols = sv_df.abs().mean().nlargest(20).index.tolist()
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(
        sample[top20_cols].T,
        cmap="coolwarm",
        center=0,
        ax=ax,
        xticklabels=False,
        cbar_kws={"label": "SHAP value"},
    )
    ax.set_title("SHAP value heatmap — top 20 features (sample of test set)")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "heatmap.png"), dpi=150)
    plt.close()

    # Step E — Local force plots (3 plots)
    cases = {
        "approve": calibrated_pds < 0.15,
        "review": (calibrated_pds >= 0.15) & (calibrated_pds < 0.35),
        "decline": calibrated_pds >= 0.35,
    }
    for case, mask in cases.items():
        matching = np.where(mask)[0]
        if len(matching) > 0:
            row_idx = matching[0]
        else:
            print(f"WARNING: no {case} example found in test set, using index 0")
            row_idx = 0

        shap_row = pd.Series(sv_array[row_idx], index=feature_names)
        top10 = shap_row.abs().nlargest(10)
        top10_vals = shap_row[top10.index]
        colors = ["#D85A30" if v > 0 else "#1D9E75" for v in top10_vals.values]

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(
            top10.index[::-1],
            top10_vals.values[::-1],
            color=colors[::-1],
        )
        ax.axvline(x=0, color="black", linewidth=0.8, linestyle="--")
        ax.set_xlabel("SHAP value")
        ax.set_title(
            f"Local explanation — {case.upper()} "
            f"(PD={calibrated_pds[row_idx]:.3f})"
        )
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"local_{case}.png"), dpi=150)
        plt.close()

    print(f"All 6 SHAP plots saved to {output_dir}")


# ──────────────────────────────────────────────────────────────────────
# Stage 3 — Eval Plots
# ──────────────────────────────────────────────────────────────────────

def plot_confusion_matrix(
    y_true: np.ndarray,
    calibrated_pds: np.ndarray,
    threshold: float = 0.35,
    output_dir: str = EVAL_PLOTS_DIR,
) -> None:
    """Plot and save a confusion matrix PNG."""
    os.makedirs(output_dir, exist_ok=True)
    from sklearn.metrics import confusion_matrix

    y_pred = (calibrated_pds >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        ax=ax,
        xticklabels=["Predicted 0", "Predicted 1"],
        yticklabels=["Actual 0", "Actual 1"],
    )
    ax.set_title(f"Confusion matrix (threshold={threshold:.2f})")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "confusion_matrix.png"), dpi=150)
    plt.close()


def plot_roc_curve(
    y_true: np.ndarray,
    calibrated_pds: np.ndarray,
    output_dir: str = EVAL_PLOTS_DIR,
) -> float:
    """Plot and save ROC curve; return AUC."""
    os.makedirs(output_dir, exist_ok=True)
    from sklearn.metrics import roc_auc_score, roc_curve

    fpr, tpr, _ = roc_curve(y_true, calibrated_pds)
    auc = float(roc_auc_score(y_true, calibrated_pds))
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fpr, tpr, color="#7F77DD", linewidth=2, label=f"AUC = {auc:.4f}")
    ax.plot([0, 1], [0, 1], color="#888780", linestyle="--", linewidth=1, label="Random")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title("ROC curve — FULL tier (test set)")
    ax.legend(loc="lower right", fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "roc_curve.png"), dpi=150)
    plt.close()
    return auc


def compute_fbeta(
    y_true: np.ndarray,
    y_pred_binary: np.ndarray,
    beta: float = 5.0,
) -> float:
    """Compute F-beta score."""
    from sklearn.metrics import fbeta_score

    return float(fbeta_score(y_true, y_pred_binary, beta=beta, zero_division=0))


def plot_fbeta_sweep(
    y_true: np.ndarray,
    calibrated_pds: np.ndarray,
    beta: float = 5.0,
    output_dir: str = EVAL_PLOTS_DIR,
) -> float:
    """Sweep thresholds 0.01–0.99 for F-beta; return best threshold."""
    os.makedirs(output_dir, exist_ok=True)
    thresholds = np.arange(0.01, 1.00, 0.01)
    fbeta_scores = []
    for t in thresholds:
        y_pred = (calibrated_pds >= t).astype(int)
        fbeta_scores.append(compute_fbeta(y_true, y_pred, beta))
    fbeta_scores = np.array(fbeta_scores)
    best_idx = int(np.argmax(fbeta_scores))
    best_threshold = float(thresholds[best_idx])
    best_fbeta = float(fbeta_scores[best_idx])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(thresholds, fbeta_scores, color="#D85A30", linewidth=2)
    ax.axvline(
        x=best_threshold,
        color="#1D9E75",
        linestyle="--",
        linewidth=1.5,
        label=f"Best threshold={best_threshold:.2f} (F{beta:.0f}={best_fbeta:.3f})",
    )
    ax.axvline(x=0.15, color="#888780", linestyle=":", linewidth=1, label="APPROVE boundary (0.15)")
    ax.axvline(x=0.35, color="#888780", linestyle=":", linewidth=1, label="DECLINE boundary (0.35)")
    ax.set_xlabel("Threshold")
    ax.set_ylabel(f"F-beta (\u03b2={beta:.0f})")
    ax.set_title(
        f"F-beta(\u03b2={beta:.0f}) sweep — diagnostic only\n"
        "F-beta(\u03b2=5) is a recall-weighted operating diagnostic. "
        "The production policy is a fixed three-band business rule."
    )
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "fbeta_sweep.png"), dpi=150)
    plt.close()
    return best_threshold


# ──────────────────────────────────────────────────────────────────────
# Entry point — unit tests
# ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 60)
    print("Stage 1 — Reason Mapping & top_5_explanations Unit Tests")
    print("=" * 60)

    # render_reason tests
    assert render_reason("CREDIT_INCOME_RATIO") == \
        "Requested credit is high relative to stated income"
    assert render_reason("BUREAU_LOAN_COUNT") == \
        "External credit history indicates elevated repayment risk"
    assert render_reason("CC_DPD_MAX") == \
        "Credit card utilization or delinquency history indicates elevated risk"
    assert render_reason("POS_RECORD_COUNT") == \
        "Past point-of-sale loan behavior indicates elevated risk"
    assert render_reason("INST_DPD_MEAN") == \
        "Past installment payments show late-payment behavior"
    assert render_reason("AGE_YEARS") == \
        "Combined application and repayment profile increased model risk"
    assert render_reason("bureau_loan_count") == \
        "Combined application and repayment profile increased model risk"
    print("  render_reason ................ PASSED")

    # top_5_explanations_from_shap tests
    test_series = pd.Series({
        "BUREAU_LOAN_COUNT":   0.42,
        "CREDIT_INCOME_RATIO": -0.31,
        "AGE_YEARS":           0.18,
        "CC_DPD_MAX":          -0.55,
        "EXT_SOURCE_MEAN":     0.67,
        "POS_RECORD_COUNT":    -0.11,
    })
    result = top_5_explanations_from_shap(test_series)
    assert len(result) == 5
    assert result[0]["feature"] == "EXT_SOURCE_MEAN"
    assert isinstance(result[0]["reason"], str)
    assert all("feature" in r and "reason" in r for r in result)
    print("  top_5_explanations_from_shap . PASSED")

    # ValueError on empty series
    try:
        top_5_explanations_from_shap(pd.Series(dtype=float))
        assert False, "Should have raised ValueError"
    except ValueError:
        pass
    print("  ValueError on empty series ... PASSED")
    print("Stage 1 unit tests passed\n")

    # ── Stage 2 — SHAP plots ────────────────────────────────────────
    print("=" * 60)
    print("Stage 2 — SHAP Plot Generation")
    print("=" * 60)

    import xgboost as xgb
    import shap

    rng = np.random.default_rng(42)
    X_synth = rng.random((300, 20)).astype(np.float32)
    y_synth = rng.integers(0, 2, 300)
    clf = xgb.XGBClassifier(n_estimators=50, random_state=42, eval_metric="logloss")
    clf.fit(X_synth, y_synth)
    explainer = shap.TreeExplainer(clf)
    feature_names = [f"feat_{i}" for i in range(20)]
    X_test_synth = rng.random((100, 20)).astype(np.float32)
    cal_pds = clf.predict_proba(X_test_synth)[:, 1]

    generate_shap_plots(
        explainer=explainer,
        X_test=X_test_synth,
        feature_names=feature_names,
        calibrated_pds=cal_pds,
        output_dir=SHAP_PLOTS_DIR,
    )

    for fname in [
        "global_summary_bar.png", "beeswarm.png", "heatmap.png",
        "local_approve.png", "local_review.png", "local_decline.png",
    ]:
        path = os.path.join(SHAP_PLOTS_DIR, fname)
        assert os.path.exists(path), f"Missing: {path}"
        assert os.path.getsize(path) > 5000, f"Too small: {path}"
    print("Stage 2 synthetic SHAP plot tests passed\n")

    # ── Stage 3 — Eval plots ────────────────────────────────────────
    print("=" * 60)
    print("Stage 3 — Eval Plots")
    print("=" * 60)

    rng2 = np.random.default_rng(99)
    y_true_synth = rng2.integers(0, 2, 300)
    pds_synth = rng2.uniform(0, 1, 300)

    plot_confusion_matrix(y_true_synth, pds_synth, threshold=0.35, output_dir=EVAL_PLOTS_DIR)
    auc = plot_roc_curve(y_true_synth, pds_synth, output_dir=EVAL_PLOTS_DIR)
    assert isinstance(auc, float)
    assert 0.0 <= auc <= 1.0

    best_t = plot_fbeta_sweep(y_true_synth, pds_synth, beta=5.0, output_dir=EVAL_PLOTS_DIR)
    assert 0.01 <= best_t <= 0.99

    for fname in ["confusion_matrix.png", "roc_curve.png", "fbeta_sweep.png"]:
        path = os.path.join(EVAL_PLOTS_DIR, fname)
        assert os.path.exists(path), f"Missing: {path}"
        assert os.path.getsize(path) > 5000
    print("Stage 3 eval plot tests passed\n")

    print("=" * 60)
    print("src/explainability.py — ALL STAGES PASSED")
    print("=" * 60)


## Module 4: Fairness Audit\n
\n
This part of Module 4 contains fairness group derivation, fairness metrics, and end-to-end fairness audit orchestration.\n
\n
Original source: `src/fairness_audit.py`\n

In [ ]:
"""Module 4 — Fairness Audit.

Proxy fairness audit across 3 group families (Primary, Secondary,
Tertiary) with 3 metrics each (DI, EOD, Brier Ratio).
Overwrites artifacts/model_fairness_audit_passed.joblib with the
final boolean result.
"""

from __future__ import annotations

import os
from typing import Any

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ──────────────────────────────────────────────────────────────────────
# Constants
# ──────────────────────────────────────────────────────────────────────

FAIRNESS_MIN_N: int = 200
FAIRNESS_MIN_DEFAULTS: int = 20
DI_THRESHOLD: float = 0.80
EOD_THRESHOLD: float = 0.10
BRIER_THRESHOLD: float = 1.25

FAIRNESS_COLS: list[str] = [
    "AMT_INCOME_TOTAL",
    "REGION_RATING_CLIENT_W_CITY",
    "NAME_INCOME_TYPE",
    "NAME_HOUSING_TYPE",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "CNT_CHILDREN",
]


def _load_or_fit_full_builder(
    train_raw: pd.DataFrame,
    raw_dir: str = "data/raw/",
    builder_path: str = "artifacts/full_feature_builder.joblib",
):
    """Boundary shim for the Module 2 FrozenFeatureBuilder contract."""
    from src.feature_engineering import FrozenFeatureBuilder, fit_full_builder

    if os.path.exists(builder_path):
        builder = joblib.load(builder_path)
        if not isinstance(builder, FrozenFeatureBuilder):
            raise TypeError(
                "full_feature_builder.joblib must contain a FrozenFeatureBuilder"
            )
        return builder

    builder = fit_full_builder(train_raw, raw_dir=raw_dir)
    if builder_path != "artifacts/full_feature_builder.joblib":
        builder.save(builder_path)
    return builder


# ──────────────────────────────────────────────────────────────────────
# Stage 4 — Fairness Group Derivation
# ──────────────────────────────────────────────────────────────────────

def derive_fairness_groups(
    df: pd.DataFrame,
    train_income_q1: float,
    train_income_q2: float,
) -> pd.DataFrame:
    """Add 8 fairness group columns using train-fitted income quantiles."""
    df = df.copy()

    # Step A — INCOME_TERTILE
    def _income_tertile(x: float) -> str:
        if pd.isna(x):
            return "MISSING"
        if x <= train_income_q1:
            return "T1"
        if x <= train_income_q2:
            return "T2"
        return "T3"

    df["INCOME_TERTILE"] = df["AMT_INCOME_TOTAL"].apply(_income_tertile)

    # Step B — FAIR_GROUP_PRIMARY
    df["FAIR_GROUP_PRIMARY"] = (
        df["INCOME_TERTILE"].astype(str)
        + "__"
        + df["REGION_RATING_CLIENT_W_CITY"].fillna(-1).astype(int).astype(str)
    )

    # Step C — INCOME_TYPE_POOLED
    from src.feature_engineering import pool_rare_categories

    df["INCOME_TYPE_POOLED"] = pool_rare_categories(
        df["NAME_INCOME_TYPE"], min_count=500
    ).fillna("MISSING")

    # Step D — HOUSING_TYPE_POOLED
    df["HOUSING_TYPE_POOLED"] = pool_rare_categories(
        df["NAME_HOUSING_TYPE"], min_count=500
    ).fillna("MISSING")

    # Step E — FAIR_GROUP_SECONDARY
    df["FAIR_GROUP_SECONDARY"] = (
        df["INCOME_TYPE_POOLED"] + "__" + df["HOUSING_TYPE_POOLED"]
    )

    # Step F — CHILDREN_BIN
    df["CHILDREN_BIN"] = pd.cut(
        df["CNT_CHILDREN"].fillna(0),
        bins=[-1, 0, 1, np.inf],
        labels=["0", "1", "2_PLUS"],
    )
    df["CHILDREN_BIN"] = df["CHILDREN_BIN"].astype(str)

    # Step G — OWN_ASSET_BIN
    df["OWN_ASSET_BIN"] = (
        df["FLAG_OWN_CAR"].fillna("N") + "_" + df["FLAG_OWN_REALTY"].fillna("N")
    )

    # Step H — FAIR_GROUP_TERTIARY
    df["FAIR_GROUP_TERTIARY"] = (
        df["OWN_ASSET_BIN"] + "__" + df["CHILDREN_BIN"]
    )

    return df


def valid_fairness_cells(
    df: pd.DataFrame,
    group_col: str,
    y_col: str = "TARGET",
    min_n: int = FAIRNESS_MIN_N,
    min_pos: int = FAIRNESS_MIN_DEFAULTS,
) -> list[str]:
    """Return group values where count >= min_n AND defaults >= min_pos."""
    stats = (
        df.groupby(group_col)[y_col]
        .agg(n="size", pos="sum")
        .reset_index()
    )
    return stats.query("n >= @min_n and pos >= @min_pos")[group_col].tolist()


# ──────────────────────────────────────────────────────────────────────
# Stage 5 — Fairness Metric Computation & Audit
# ──────────────────────────────────────────────────────────────────────

def compute_fairness_metrics(
    df: pd.DataFrame,
    group_col: str,
    y_col: str = "TARGET",
    pd_col: str = "CALIBRATED_PD",
    decision_col: str = "DECISION",
    min_n: int = FAIRNESS_MIN_N,
    min_pos: int = FAIRNESS_MIN_DEFAULTS,
) -> pd.DataFrame:
    """Compute DI, EOD, Brier ratio for each group cell."""
    eligible = valid_fairness_cells(df, group_col, y_col, min_n, min_pos)

    rows: list[dict[str, Any]] = []
    for gval in df[group_col].unique():
        sub = df[df[group_col] == gval]
        n = len(sub)
        n_defaults = int(sub[y_col].sum())
        evaluable = gval in eligible

        if not evaluable:
            rows.append({
                "group_value": str(gval),
                "n": n,
                "n_defaults": n_defaults,
                "approve_rate": np.nan,
                "tpr": np.nan,
                "brier_score": np.nan,
                "di_ratio": np.nan,
                "eod": np.nan,
                "brier_ratio": np.nan,
                "evaluable": False,
                "di_pass": np.nan,
                "eod_pass": np.nan,
                "brier_pass": np.nan,
            })
            continue

        approve_rate = float((sub[decision_col] == "APPROVE").mean())
        good = sub[sub[y_col] == 0]
        tpr = float(
            (good[decision_col] == "APPROVE").mean() if len(good) > 0 else np.nan
        )
        brier = float(np.mean((sub[pd_col].values - sub[y_col].values) ** 2))

        rows.append({
            "group_value": str(gval),
            "n": n,
            "n_defaults": n_defaults,
            "approve_rate": approve_rate,
            "tpr": tpr,
            "brier_score": brier,
            "di_ratio": np.nan,
            "eod": np.nan,
            "brier_ratio": np.nan,
            "evaluable": True,
            "di_pass": np.nan,
            "eod_pass": np.nan,
            "brier_pass": np.nan,
        })

    result = pd.DataFrame(rows)
    eval_mask = result["evaluable"] == True  # noqa: E712

    if eval_mask.sum() == 0:
        return result

    # Compute family-level metrics across evaluable cells
    max_approve = result.loc[eval_mask, "approve_rate"].max()
    min_approve = result.loc[eval_mask, "approve_rate"].min()
    max_tpr = result.loc[eval_mask, "tpr"].max()
    min_tpr = result.loc[eval_mask, "tpr"].min()
    max_brier = result.loc[eval_mask, "brier_score"].max()
    min_brier = result.loc[eval_mask, "brier_score"].min()

    family_di_pass = (
        (min_approve / max_approve >= DI_THRESHOLD) if max_approve > 0 else False
    )
    family_eod_pass = (max_tpr - min_tpr) <= EOD_THRESHOLD
    family_brier_pass = (
        (max_brier / min_brier <= BRIER_THRESHOLD) if min_brier > 0 else False
    )

    result.loc[eval_mask, "di_ratio"] = (
        result.loc[eval_mask, "approve_rate"] / max_approve
    )
    result.loc[eval_mask, "eod"] = result.loc[eval_mask, "tpr"] - max_tpr
    result.loc[eval_mask, "brier_ratio"] = (
        result.loc[eval_mask, "brier_score"] / min_brier if min_brier > 0 else np.nan
    )
    result.loc[eval_mask, "di_pass"] = family_di_pass
    result.loc[eval_mask, "eod_pass"] = family_eod_pass
    result.loc[eval_mask, "brier_pass"] = family_brier_pass

    return result


def _plot_fairness_summary_card(
    primary_passes: dict[str, bool],
    secondary_passes: dict[str, bool],
    tertiary_passes: dict[str, bool],
    output_dir: str,
) -> None:
    """Produce a 3×3 grid PNG showing pass/fail for each metric × family."""
    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    families = ["Primary", "Secondary", "Tertiary"]
    passes_list = [primary_passes, secondary_passes, tertiary_passes]
    metrics = ["DI (≥0.80)", "EOD (≤0.10)", "Brier (≤1.25)"]
    metric_keys = ["di", "eod", "brier"]

    for i, (ax, family, passes) in enumerate(zip(axes, families, passes_list)):
        ax.set_title(f"{family} group", fontsize=11)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis("off")
        for j, (label, key) in enumerate(zip(metrics, metric_keys)):
            passed = passes.get(key, False)
            color = "#1D9E75" if passed else "#D85A30"
            symbol = "PASS" if passed else "FAIL"
            y_pos = 0.75 - j * 0.28
            ax.add_patch(
                plt.Rectangle(
                    (0.05, y_pos - 0.08), 0.9, 0.22,
                    color=color, alpha=0.15,
                )
            )
            ax.text(
                0.5, y_pos + 0.03,
                f"{label}: {symbol}",
                ha="center", va="center",
                fontsize=9, color=color, fontweight="bold",
            )

    plt.suptitle(
        "Proxy fairness audit — pass/fail summary\n"
        "These metrics evaluate stability across proxy-defined "
        "subgroups only and must not be interpreted as proof of "
        "fairness across legally protected characteristics.",
        fontsize=8, y=1.02,
    )
    plt.tight_layout()
    os.makedirs(output_dir, exist_ok=True)
    plt.savefig(
        os.path.join(output_dir, "fairness_summary_card.png"),
        dpi=150, bbox_inches="tight",
    )
    plt.close()


def _family_passes(audit_df: pd.DataFrame) -> dict[str, bool]:
    """Extract family-level pass/fail from audit DataFrame."""
    ev = audit_df[audit_df["evaluable"] == True]  # noqa: E712
    if len(ev) == 0:
        return {"di": False, "eod": False, "brier": False}
    di_val = ev["di_pass"].iloc[0]
    eod_val = ev["eod_pass"].iloc[0]
    brier_val = ev["brier_pass"].iloc[0]
    return {
        "di": bool(di_val),
        "eod": bool(eod_val),
        "brier": bool(brier_val),
    }


def run_full_fairness_audit(
    df: pd.DataFrame,
    train_income_q1: float,
    train_income_q2: float,
    output_dir: str,
) -> bool:
    """Run full 3-family fairness audit; return True if all pass."""
    os.makedirs(output_dir, exist_ok=True)

    # Step A — derive groups
    df = derive_fairness_groups(df, train_income_q1, train_income_q2)

    # Step B — compute metrics per family
    primary_df = compute_fairness_metrics(df, "FAIR_GROUP_PRIMARY")
    secondary_df = compute_fairness_metrics(df, "FAIR_GROUP_SECONDARY")
    tertiary_df = compute_fairness_metrics(df, "FAIR_GROUP_TERTIARY")

    # Step C — save CSVs
    primary_df.to_csv(
        os.path.join(output_dir, "audit_primary.csv"), index=False
    )
    secondary_df.to_csv(
        os.path.join(output_dir, "audit_secondary.csv"), index=False
    )
    tertiary_df.to_csv(
        os.path.join(output_dir, "audit_tertiary.csv"), index=False
    )

    # Step D — extract family-level pass/fail
    p_passes = _family_passes(primary_df)
    s_passes = _family_passes(secondary_df)
    t_passes = _family_passes(tertiary_df)

    # Step E — summary card
    _plot_fairness_summary_card(p_passes, s_passes, t_passes, output_dir)

    # Step F — print audit results
    print("\n=== Proxy Fairness Audit Results ===")
    for name, passes in [
        ("Primary", p_passes),
        ("Secondary", s_passes),
        ("Tertiary", t_passes),
    ]:
        print(
            f"{name}: DI={'PASS' if passes['di'] else 'FAIL'}  "
            f"EOD={'PASS' if passes['eod'] else 'FAIL'}  "
            f"Brier={'PASS' if passes['brier'] else 'FAIL'}"
        )

    # Step G — determine overall result
    all_pass = all([
        p_passes["di"], p_passes["eod"], p_passes["brier"],
        s_passes["di"], s_passes["eod"], s_passes["brier"],
        t_passes["di"], t_passes["eod"], t_passes["brier"],
    ])
    print(f"\nmodel_fairness_audit_passed = {all_pass}")
    return all_pass


# ──────────────────────────────────────────────────────────────────────
# main() orchestration
# ──────────────────────────────────────────────────────────────────────

def main() -> None:
    """Module 4 orchestration — detects real vs synthetic data."""
    DATA_PROCESSED_DIR = os.environ.get("DATA_PROCESSED_DIR", "data/processed/")
    ARTIFACT_DIR = os.environ.get("ARTIFACT_DIR", "artifacts/")
    FAIRNESS_PLOTS_DIR = os.environ.get("FAIRNESS_PLOTS_DIR", "notebooks/fairness_plots/")
    SHAP_PLOTS_DIR = os.environ.get("SHAP_PLOTS_DIR", "notebooks/shap_plots/")
    EVAL_PLOTS_DIR = os.environ.get("EVAL_PLOTS_DIR", "notebooks/eval_plots/")

    val_policy_path = os.path.join(DATA_PROCESSED_DIR, "val_policy.pkl")
    real_data = os.path.exists(val_policy_path)
    print(f"[Module 4 Real Data Audit] Checking for real data at '{val_policy_path}' — found: {real_data}")
    print(f"[Module 4] FORCING REAL DATA MODE per user directive")
    real_data = True  # FORCE REAL MODE

    if real_data:
        import pickle
        import sys

        sys.path.insert(0, ".")

        print("[M4] Loading pickles...")
        try:
            with open(os.path.join(DATA_PROCESSED_DIR, "train.pkl"), "rb") as f:
                train_raw = pickle.load(f)
            print(f"[M4]   ✓ train.pkl loaded: {train_raw.shape}")
            with open(os.path.join(DATA_PROCESSED_DIR, "test.pkl"), "rb") as f:
                test_raw = pickle.load(f)
            print(f"[M4]   ✓ test.pkl loaded: {test_raw.shape}")
        except Exception as e:
            print(f"[M4] ✗ Error loading pickles: {e}")
            raise

        print("[M4] Loading modules...")
        try:
            from src.feature_engineering import build_full
            from src.models.train import decision_from_pd, load_artifacts
            print("[M4]   ✓ Modules imported")
        except Exception as e:
            print(f"[M4] ✗ Error importing modules: {e}")
            raise

        print("[M4] Loading artifacts...")
        try:
            artifacts = load_artifacts(ARTIFACT_DIR)
            print(f"[M4]   ✓ Artifacts loaded")

            model = artifacts.get("full_model")
            calibrator = artifacts.get("full_calibrator")
            explainer = artifacts.get("full_shap_explainer")
            print(f"[M4]   ✓ model={type(model).__name__}, calibrator={type(calibrator).__name__}, explainer={type(explainer).__name__}")
        except Exception as e:
            print(f"[M4] ✗ Error loading artifacts: {e}")
            raise

        print("[M4] Loading feature builder...")
        try:
            full_builder = _load_or_fit_full_builder(
                train_raw,
                raw_dir="data/raw/",
                builder_path="artifacts/full_feature_builder.joblib",
            )
            print(f"[M4]   ✓ Feature builder loaded")
        except Exception as e:
            print(f"[M4] ✗ Error loading feature builder: {e}")
            raise

        print("[M4] Building features (this may take 1-2 minutes)...")
        try:
            X_test_full = build_full(test_raw, full_builder, raw_dir="data/raw/")
            print(f"[M4]   ✓ Features built: {X_test_full.shape}")
            feature_names = list(X_test_full.columns)
            X_test_arr = X_test_full.values
        except Exception as e:
            print(f"[M4] ✗ Error building features: {e}")
            raise

        if model is None or calibrator is None or explainer is None:
            print("WARNING: Missing Module 3 artifacts; training fallback model for Module 4 execution.")
            import xgboost as xgb
            from sklearn.calibration import CalibratedClassifierCV
            import shap

            X_train_full = build_full(train_raw, full_builder, raw_dir="data/raw/")
            y_train = train_raw["TARGET"].values

            model = xgb.XGBClassifier(
                n_estimators=100,
                random_state=42,
                eval_metric="logloss",
                use_label_encoder=False,
            )
            model.fit(X_train_full.values, y_train)

            if calibrator is None:
                calibrator = CalibratedClassifierCV(base_estimator=model, cv=3, method="sigmoid")
                calibrator.fit(X_train_full.values, y_train)

            if explainer is None:
                explainer = shap.TreeExplainer(model)

        raw_pds = model.predict_proba(X_test_arr)[:, 1]
        cal_pds = calibrator.predict(raw_pds)
        decisions = np.array([decision_from_pd(p) for p in cal_pds])

        # Build audit DataFrame from raw test columns
        fairness_cols_present = [c for c in FAIRNESS_COLS if c in test_raw.columns]
        test_audit_df = test_raw[["TARGET"] + fairness_cols_present].copy()
        test_audit_df["CALIBRATED_PD"] = cal_pds
        test_audit_df["DECISION"] = decisions

        train_income_q1 = float(train_raw["AMT_INCOME_TOTAL"].quantile(1 / 3))
        train_income_q2 = float(train_raw["AMT_INCOME_TOTAL"].quantile(2 / 3))

        audit_passed = run_full_fairness_audit(
            test_audit_df, train_income_q1, train_income_q2, FAIRNESS_PLOTS_DIR
        )

        from src.explainability import (
            generate_shap_plots,
            plot_confusion_matrix,
            plot_fbeta_sweep,
            plot_roc_curve,
        )

        generate_shap_plots(
            explainer, X_test_arr, feature_names, cal_pds, SHAP_PLOTS_DIR
        )
        plot_confusion_matrix(
            test_raw["TARGET"].values, cal_pds, threshold=0.35, output_dir=EVAL_PLOTS_DIR
        )
        plot_roc_curve(
            test_raw["TARGET"].values, cal_pds, output_dir=EVAL_PLOTS_DIR
        )
        plot_fbeta_sweep(
            test_raw["TARGET"].values, cal_pds, beta=5.0, output_dir=EVAL_PLOTS_DIR
        )

    else:
        print("Real data not found — using synthetic mode")
        import xgboost as xgb
        import shap

        rng = np.random.default_rng(42)
        X_synth = rng.random((500, 90)).astype(np.float32)
        y_synth = rng.integers(0, 2, 500)
        clf = xgb.XGBClassifier(
            n_estimators=50, random_state=42, eval_metric="logloss"
        )
        clf.fit(X_synth, y_synth)
        explainer = shap.TreeExplainer(clf)
        feature_names = [f"feat_{i}" for i in range(90)]
        X_test_arr = rng.random((200, 90)).astype(np.float32)
        cal_pds = clf.predict_proba(X_test_arr)[:, 1]
        decisions = np.array([
            "APPROVE" if p < 0.15 else "REVIEW" if p < 0.35 else "DECLINE"
            for p in cal_pds
        ])

        synth_income = rng.uniform(10000, 500000, 200)
        q1 = float(np.percentile(synth_income, 100 / 3))
        q2 = float(np.percentile(synth_income, 200 / 3))
        audit_df = pd.DataFrame({
            "TARGET": rng.integers(0, 2, 200),
            "AMT_INCOME_TOTAL": synth_income,
            "REGION_RATING_CLIENT_W_CITY": rng.choice([1, 2, 3], 200),
            "NAME_INCOME_TYPE": rng.choice(
                ["Working", "Pensioner", "Commercial associate",
                 "State servant", "Unemployed"], 200
            ),
            "NAME_HOUSING_TYPE": rng.choice(
                ["House / apartment", "Rented apartment",
                 "With parents", "Municipal apartment"], 200
            ),
            "FLAG_OWN_CAR": rng.choice(["Y", "N"], 200),
            "FLAG_OWN_REALTY": rng.choice(["Y", "N"], 200),
            "CNT_CHILDREN": rng.integers(0, 5, 200).astype(float),
            "CALIBRATED_PD": cal_pds,
            "DECISION": decisions,
        })

        audit_passed = run_full_fairness_audit(
            audit_df, q1, q2, FAIRNESS_PLOTS_DIR
        )

        from src.explainability import (
            generate_shap_plots,
            plot_confusion_matrix,
            plot_fbeta_sweep,
            plot_roc_curve,
        )

        generate_shap_plots(
            explainer, X_test_arr, feature_names, cal_pds, SHAP_PLOTS_DIR
        )
        plot_confusion_matrix(
            audit_df["TARGET"].values, cal_pds, threshold=0.35, output_dir=EVAL_PLOTS_DIR
        )
        plot_roc_curve(
            audit_df["TARGET"].values, cal_pds, output_dir=EVAL_PLOTS_DIR
        )
        plot_fbeta_sweep(
            audit_df["TARGET"].values, cal_pds, beta=5.0, output_dir=EVAL_PLOTS_DIR
        )

    # Overwrite model_fairness_audit_passed.joblib
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    joblib.dump(
        bool(audit_passed),
        os.path.join(ARTIFACT_DIR, "model_fairness_audit_passed.joblib"),
    )
    print(f"model_fairness_audit_passed.joblib overwritten with: {audit_passed}")
    print("Module 4 complete")


# ──────────────────────────────────────────────────────────────────────
# Entry point
# ──────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # ── Unit tests first ─────────────────────────────────────────────
    print("=" * 60)
    print("Stage 4 — Fairness Group Derivation Tests")
    print("=" * 60)

    rng = np.random.default_rng(1)
    synth_df = pd.DataFrame({
        "AMT_INCOME_TOTAL": np.random.default_rng(1).uniform(10000, 500000, 1000),
        "REGION_RATING_CLIENT_W_CITY": np.random.default_rng(2).choice([1, 2, 3], 1000),
        "NAME_INCOME_TYPE": np.random.default_rng(3).choice(
            ["Working", "Pensioner", "Commercial associate",
             "State servant", "Unemployed"], 1000
        ),
        "NAME_HOUSING_TYPE": np.random.default_rng(4).choice(
            ["House / apartment", "Rented apartment",
             "With parents", "Municipal apartment"], 1000
        ),
        "FLAG_OWN_CAR": np.random.default_rng(5).choice(["Y", "N"], 1000),
        "FLAG_OWN_REALTY": np.random.default_rng(6).choice(["Y", "N"], 1000),
        "CNT_CHILDREN": np.random.default_rng(7).integers(0, 5, 1000).astype(float),
        "TARGET": np.random.default_rng(8).integers(0, 2, 1000),
    })

    q1 = float(np.percentile(synth_df["AMT_INCOME_TOTAL"], 100 / 3))
    q2 = float(np.percentile(synth_df["AMT_INCOME_TOTAL"], 200 / 3))
    result = derive_fairness_groups(synth_df, q1, q2)

    for col in [
        "INCOME_TERTILE", "FAIR_GROUP_PRIMARY",
        "INCOME_TYPE_POOLED", "HOUSING_TYPE_POOLED",
        "FAIR_GROUP_SECONDARY", "CHILDREN_BIN",
        "OWN_ASSET_BIN", "FAIR_GROUP_TERTIARY",
    ]:
        assert col in result.columns, f"Missing column: {col}"

    assert set(result["INCOME_TERTILE"].unique()).issubset({"T1", "T2", "T3", "MISSING"})
    assert result["FAIR_GROUP_PRIMARY"].str.contains("__").all()
    assert result["CHILDREN_BIN"].isin(["0", "1", "2_PLUS"]).all()
    print("  derive_fairness_groups ......... PASSED")

    cells = valid_fairness_cells(result, "FAIR_GROUP_PRIMARY", min_n=50, min_pos=5)
    assert isinstance(cells, list)
    assert len(cells) >= 1
    print("  valid_fairness_cells ........... PASSED")
    print("Stage 4 tests passed\n")

    # ── Stage 5 — Run full audit synthetic test ──────────────────────
    print("=" * 60)
    print("Stage 5 — Full Fairness Audit Synthetic Test")
    print("=" * 60)

    # Add CALIBRATED_PD and DECISION columns for audit
    synth_df_audit = synth_df.copy()
    synth_df_audit["CALIBRATED_PD"] = np.random.default_rng(9).uniform(0, 1, 1000)
    synth_df_audit["DECISION"] = np.where(
        synth_df_audit["CALIBRATED_PD"] < 0.15, "APPROVE",
        np.where(synth_df_audit["CALIBRATED_PD"] < 0.35, "REVIEW", "DECLINE"),
    )

    FAIRNESS_PLOTS_DIR = os.environ.get("FAIRNESS_PLOTS_DIR", "notebooks/fairness_plots/")
    audit_passed = run_full_fairness_audit(
        synth_df_audit, q1, q2, FAIRNESS_PLOTS_DIR
    )
    assert isinstance(audit_passed, bool)

    # Verify CSVs
    for fname in ["audit_primary.csv", "audit_secondary.csv", "audit_tertiary.csv"]:
        path = os.path.join(FAIRNESS_PLOTS_DIR, fname)
        assert os.path.exists(path), f"Missing: {path}"
        csv_df = pd.read_csv(path)
        expected_cols = [
            "group_value", "n", "n_defaults", "approve_rate", "tpr",
            "brier_score", "di_ratio", "eod", "brier_ratio",
            "evaluable", "di_pass", "eod_pass", "brier_pass",
        ]
        assert list(csv_df.columns) == expected_cols, f"Column mismatch in {fname}"
    print("  Audit CSVs verified ............ PASSED")

    # Verify summary card
    card_path = os.path.join(FAIRNESS_PLOTS_DIR, "fairness_summary_card.png")
    assert os.path.exists(card_path), "Missing fairness_summary_card.png"
    assert os.path.getsize(card_path) > 5000
    print("  fairness_summary_card.png ...... PASSED")

    print("Stage 5 tests passed\n")

    # ── Run main() in synthetic mode ─────────────────────────────────
    print("=" * 60)
    print("Running main() — synthetic mode")
    print("=" * 60)
    main()


## Module 5: Flask API\n
\n
Module 5 contains the strict Flask scoring API, payload validation, routing logic, runtime loading, and UI configuration helpers.\n
\n
Original source: `src/api/app.py`\n

In [ ]:
"""Module 5 - strict Flask scoring API."""

from __future__ import annotations

from dataclasses import dataclass
import importlib
import inspect
import json
import os
from pathlib import Path
import re
import tempfile
from types import MappingProxyType, ModuleType
from typing import Any, Mapping

from flask import Flask, jsonify, render_template, request
import joblib
import numpy as np
import pandas as pd

from configs.config import (
    APPROVE_THRESHOLD,
    ARTIFACT_DIR,
    DATA_DIR,
    DECLINE_THRESHOLD,
    FAIRNESS_AUDIT_VERSION,
    FULL_REQUIRED_SECTIONS,
    MODEL_VERSIONS,
)
from src.explainability import render_reason, top_5_explanations_from_shap
from src.runtime_verification import validate_transformed_frame

ROUTER_VERSION = "router_v1.0.0"
POLICY_VERSION = "policy_v1.0.0"
DEFAULT_FAIRNESS_VERSION_TAG = "2026Q1"
RUNTIME_EXTENSION_KEY = "mastermind_runtime"
PROCESSED_MANIFEST_FILENAME = "processed_artifact_manifest.json"
REPRODUCIBILITY_REPORT_FILENAME = "reproducibility_report.json"
FAIRNESS_RESULT_FILENAME = "model_fairness_audit_passed.joblib"
UI_DEFAULT_TIER = "REDUCED"
PROJECT_ROOT = Path(__file__).resolve().parents[2]
TEMPLATE_DIR = PROJECT_ROOT / "templates"
STATIC_DIR = PROJECT_ROOT / "static"
CATEGORICAL_APPLICATION_FIELDS = frozenset(
    {
        "NAME_CONTRACT_TYPE",
        "NAME_TYPE_SUITE",
        "NAME_EDUCATION_TYPE",
        "NAME_FAMILY_STATUS",
        "OCCUPATION_TYPE",
        "ORGANIZATION_TYPE",
        "WEEKDAY_APPR_PROCESS_START",
    }
)
DEMO_FIELD_OPTIONS: Mapping[str, tuple[str, ...]] = MappingProxyType(
    {
        "NAME_CONTRACT_TYPE": ("Cash loans", "Revolving loans"),
        "NAME_TYPE_SUITE": ("Unaccompanied", "Family", "Spouse, partner"),
        "NAME_EDUCATION_TYPE": (
            "Higher education",
            "Secondary / secondary special",
            "Incomplete higher",
        ),
        "NAME_FAMILY_STATUS": ("Married", "Single / not married", "Civil marriage"),
        "OCCUPATION_TYPE": ("Laborers", "Core staff", "Sales staff"),
        "ORGANIZATION_TYPE": (
            "Business Entity Type 3",
            "Self-employed",
            "School",
        ),
        "WEEKDAY_APPR_PROCESS_START": (
            "MONDAY",
            "TUESDAY",
            "WEDNESDAY",
            "THURSDAY",
            "FRIDAY",
        ),
    }
)

# Frozen public API contract for request payloads.
APPLICATION_REQUIRED_FIELDS: tuple[str, ...] = (
    "AMT_INCOME_TOTAL_CAPPED",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "DAYS_LAST_PHONE_CHANGE",
    "REGION_POPULATION_RELATIVE",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "CNT_FAM_MEMBERS",
    "OWN_CAR_AGE",
    "OBS_30_CNT_SOCIAL_CIRCLE",
    "DEF_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE",
    "DEF_60_CNT_SOCIAL_CIRCLE",
    "AMT_REQ_CREDIT_BUREAU_HOUR",
    "AMT_REQ_CREDIT_BUREAU_DAY",
    "AMT_REQ_CREDIT_BUREAU_WEEK",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
    "NAME_CONTRACT_TYPE",
    "NAME_TYPE_SUITE",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
    "WEEKDAY_APPR_PROCESS_START",
    "DAYS_EMPLOYED_ANOM",
)
AGG_REQUIRED_FIELDS: Mapping[str, tuple[str, ...]] = MappingProxyType(
    {
        "bureau_agg": (
            "BUREAU_LOAN_COUNT",
            "BUREAU_ACTIVE_COUNT",
            "BUREAU_CLOSED_COUNT",
            "BUREAU_AMT_CREDIT_SUM_SUM",
            "BUREAU_AMT_CREDIT_SUM_DEBT_SUM",
            "BUREAU_DEBT_TO_CREDIT_RATIO",
            "BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM",
            "BUREAU_CREDIT_DAY_OVERDUE_MAX",
            "BUREAU_DAYS_CREDIT_MAX",
            "BUREAU_CNT_CREDIT_PROLONG_SUM",
        ),
        "previous_agg": (
            "PREV_APP_COUNT",
            "PREV_APPROVED_COUNT",
            "PREV_REFUSED_COUNT",
            "PREV_APPROVAL_RATE",
            "PREV_REFUSAL_RATE",
            "PREV_AMT_APPLICATION_MEAN",
            "PREV_AMT_CREDIT_MEAN",
            "PREV_AMT_GOODS_PRICE_MEAN",
            "PREV_APP_CREDIT_DIFF_MEAN",
            "PREV_DAYS_DECISION_MAX",
            "PREV_RATE_DOWN_PAYMENT_MEAN",
        ),
        "installments_agg": (
            "INST_RECORD_COUNT",
            "INST_MISSED_RATE",
            "INST_DPD_MEAN",
            "INST_DPD_MAX",
            "INST_PAYMENT_RATIO_MEAN",
            "INST_PAYMENT_RATIO_MIN",
            "INST_LATE_COUNT",
        ),
        "pos_cash_agg": (
            "POS_RECORD_COUNT",
            "POS_DPD_MEAN",
            "POS_DPD_MAX",
            "POS_DPD_DEF_MEAN",
            "POS_DPD_DEF_MAX",
            "POS_COMPLETED_RATE",
            "POS_ACTIVE_RATE",
            "POS_CNT_INSTALMENT_FUTURE_MEAN",
        ),
        "credit_card_agg": (
            "CC_RECORD_COUNT",
            "CC_BALANCE_MEAN",
            "CC_LIMIT_MEAN",
            "CC_UTILIZATION_MEAN",
            "CC_PAYMENT_RATIO_MEAN",
            "CC_DPD_MEAN",
            "CC_DPD_MAX",
            "CC_DRAWINGS_ATM_SUM",
            "CC_DRAWINGS_CURRENT_SUM",
        ),
    }
)
ALLOWED_TOP_LEVEL_KEYS = frozenset(FULL_REQUIRED_SECTIONS)
FULL_SECTION_ORDER: tuple[str, ...] = (
    "application",
    "bureau_agg",
    "previous_agg",
    "installments_agg",
    "pos_cash_agg",
    "credit_card_agg",
)
FULL_ONLY_SECTION_ORDER: tuple[str, ...] = FULL_SECTION_ORDER[1:]
FULL_ONLY_SECTIONS = frozenset(FULL_ONLY_SECTION_ORDER)

ERROR_MESSAGES: Mapping[str, str] = MappingProxyType(
    {
        "bad_request": "Bad request.",
        "missing_application": "Application section is required.",
        "forbidden_field_code_gender": "CODE_GENDER is not allowed.",
        "partial_full_payload_not_allowed": "Partial FULL payloads are not allowed.",
        "missing_application_fields": "Application payload is missing required fields.",
        "missing_aggregate_fields": "FULL payload is missing required aggregate fields.",
        "starter_not_supported_in_mvp": "Payload is not supported in MVP.",
        "internal_error": "Scoring failed.",
    }
)


@dataclass(frozen=True)
class ApiRuntime:
    """Immutable runtime loaded once at startup."""

    artifact_dir: str
    processed_dir: str
    processed_manifest: Mapping[str, Any]
    full_builder: Any
    full_model: Any
    full_calibrator: Any
    full_shap_explainer: Any
    reduced_builder: Any
    reduced_model: Any
    reduced_calibrator: Any
    reduced_shap_explainer: Any
    model_fairness_audit_passed: bool
    health_model_version: str
    tier_model_versions: Mapping[str, str]
    coverage_tiers_available: tuple[str, ...]
    reproducibility_report: Mapping[str, Any]
    mock_mode: bool


@dataclass(frozen=True)
class ApiError(Exception):
    """Structured API error for stable JSON responses."""

    status_code: int
    error_code: str
    message: str
    missing_fields: tuple[str, ...] = ()

    def to_response(self):
        payload = {
            "error_code": self.error_code,
            "message": self.message,
        }
        if self.status_code in (400, 422):
            payload["missing_fields"] = list(self.missing_fields)
        return jsonify(payload), self.status_code


class _MockBuilder:
    def __init__(self, tier: str):
        self.tier = tier.upper()
        base_columns = [
            "mock_income",
            "mock_credit",
            "mock_annuity",
            "mock_ratio",
            "mock_ext_mean",
            "mock_social",
            "mock_bureau",
            "mock_previous",
        ]
        if self.tier == "FULL":
            base_columns.extend(["mock_installments", "mock_pos", "mock_cc"])
        self.encoded_columns_ = base_columns

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        out = pd.DataFrame(index=df.index)
        income = _numeric_series(df, "AMT_INCOME_TOTAL_CAPPED")
        credit = _numeric_series(df, "AMT_CREDIT")
        annuity = _numeric_series(df, "AMT_ANNUITY")
        ext1 = _numeric_series(df, "EXT_SOURCE_1")
        ext2 = _numeric_series(df, "EXT_SOURCE_2")
        ext3 = _numeric_series(df, "EXT_SOURCE_3")
        social = (
            _numeric_series(df, "OBS_30_CNT_SOCIAL_CIRCLE")
            + _numeric_series(df, "DEF_30_CNT_SOCIAL_CIRCLE")
            + _numeric_series(df, "OBS_60_CNT_SOCIAL_CIRCLE")
            + _numeric_series(df, "DEF_60_CNT_SOCIAL_CIRCLE")
        )
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where(income.to_numpy() > 0, credit.to_numpy() / income.to_numpy(), 0.0)

        out["mock_income"] = income.astype(float)
        out["mock_credit"] = credit.astype(float)
        out["mock_annuity"] = annuity.astype(float)
        out["mock_ratio"] = ratio.astype(float)
        out["mock_ext_mean"] = ((ext1 + ext2 + ext3) / 3.0).astype(float)
        out["mock_social"] = social.astype(float)
        out["mock_bureau"] = _numeric_series(df, "BUREAU_LOAN_COUNT")
        out["mock_previous"] = _numeric_series(df, "PREV_APP_COUNT")
        if self.tier == "FULL":
            out["mock_installments"] = _numeric_series(df, "INST_RECORD_COUNT")
            out["mock_pos"] = _numeric_series(df, "POS_RECORD_COUNT")
            out["mock_cc"] = _numeric_series(df, "CC_RECORD_COUNT")
        return out[self.encoded_columns_].astype(float)


class _MockModel:
    def __init__(self, feature_count: int):
        self.n_features_in_ = feature_count

    def predict_proba(self, X: Any) -> np.ndarray:
        arr = _coerce_2d_numeric(X)
        if arr.shape[1] != self.n_features_in_:
            raise ValueError(
                f"Expected {self.n_features_in_} features but received {arr.shape[1]}"
            )
        weights = np.linspace(0.15, 0.65, arr.shape[1], dtype=float)
        score = arr @ weights / max(arr.shape[1], 1)
        prob = 1.0 / (1.0 + np.exp(-(score / 100000.0)))
        prob = np.clip(prob, 0.01, 0.99)
        return np.column_stack([1.0 - prob, prob])


class _MockCalibrator:
    def predict(self, raw_pd: Any) -> np.ndarray:
        arr = np.asarray(raw_pd, dtype=float).reshape(-1)
        calibrated = 0.92 * arr + 0.03
        return np.clip(calibrated, 0.0, 1.0)


class _MockExplainer:
    def __call__(self, X: Any) -> np.ndarray:
        arr = _coerce_2d_numeric(X)
        weights = np.linspace(1.0, 2.0, arr.shape[1], dtype=float)
        return arr * weights


def create_app(
    artifact_dir: str | None = None,
    processed_dir: str | None = None,
    mock_mode: bool = False,
    strict_artifacts: bool = True,
) -> Flask:
    """Create the Flask app with eager startup validation."""

    resolved_artifact_dir, resolved_processed_dir = _resolve_runtime_dirs(
        artifact_dir,
        processed_dir,
        mock_mode=mock_mode,
    )

    runtime = (
        _build_mock_runtime(resolved_artifact_dir, resolved_processed_dir)
        if mock_mode
        else _load_real_runtime(
            artifact_dir=resolved_artifact_dir,
            processed_dir=resolved_processed_dir,
            strict_artifacts=strict_artifacts,
        )
    )

    app = Flask(
        __name__,
        template_folder=str(TEMPLATE_DIR),
        static_folder=str(STATIC_DIR),
    )
    app.extensions[RUNTIME_EXTENSION_KEY] = runtime

    @app.get("/")
    def home():
        loaded_runtime = _get_runtime(app)
        return render_template(
            "index.html",
            page_title="MasterMind Credit Scoring",
            active_nav="home",
            ui_config=_build_demo_config(loaded_runtime),
            health_snapshot=_build_health_snapshot(loaded_runtime),
        )

    @app.get("/analyze")
    @app.get("/demo")
    def analyze():
        loaded_runtime = _get_runtime(app)
        return render_template(
            "analyze.html",
            page_title="Credit Analysis",
            active_nav="analyze",
            ui_config=_build_demo_config(loaded_runtime),
            health_snapshot=_build_health_snapshot(loaded_runtime),
        )

    @app.get("/status")
    def status_page():
        loaded_runtime = _get_runtime(app)
        return render_template(
            "status.html",
            page_title="System Status",
            active_nav="status",
            ui_config=_build_demo_config(loaded_runtime),
            health_snapshot=_build_health_snapshot(loaded_runtime),
        )

    @app.get("/health")
    def health():
        loaded_runtime = _get_runtime(app)
        return jsonify(_build_health_snapshot(loaded_runtime))

    @app.post("/score")
    def score():
        try:
            try:
                payload = request.get_json(silent=False)
            except Exception as exc:  # pragma: no cover - Flask wraps malformed JSON differently by version
                raise _api_error("bad_request") from exc

            if payload is None or not isinstance(payload, dict):
                raise _api_error("bad_request")

            error_code, missing_fields = validate_payload(payload)
            if error_code is not None:
                raise _api_error(error_code, missing_fields)

            response = score_request(payload, _get_runtime(app), mock_mode=mock_mode)
            return jsonify(response)
        except ApiError as exc:
            return exc.to_response()
        except Exception:
            app.logger.exception("Scoring failed.")
            return jsonify(
                {
                    "error_code": "internal_error",
                    "message": ERROR_MESSAGES["internal_error"],
                }
            ), 500

    return app


def validate_payload(payload: dict) -> tuple[str | None, list[str]]:
    """Validate the public JSON contract and return an error code if invalid."""

    if not isinstance(payload, dict):
        return "bad_request", []

    unexpected_top_keys = sorted(set(payload) - ALLOWED_TOP_LEVEL_KEYS)
    if unexpected_top_keys:
        return "bad_request", []

    application = payload.get("application")
    if application is None:
        return "missing_application", ["application"]
    if not isinstance(application, dict):
        return "bad_request", []

    for section_name, section_value in payload.items():
        if not isinstance(section_value, dict):
            return "bad_request", []
        if not _section_has_scalar_values(section_value):
            return "bad_request", []

    if "CODE_GENDER" in application:
        return "forbidden_field_code_gender", []

    try:
        tier = determine_coverage_tier(payload)
    except ValueError:
        if set(payload).intersection(FULL_ONLY_SECTIONS):
            missing_sections = sorted(set(FULL_SECTION_ORDER) - set(payload))
            return "partial_full_payload_not_allowed", missing_sections
        return "starter_not_supported_in_mvp", []

    missing_application_fields = sorted(
        field for field in APPLICATION_REQUIRED_FIELDS if field not in application
    )
    if missing_application_fields:
        return "missing_application_fields", missing_application_fields

    if tier == "FULL":
        missing_aggregate_fields: list[str] = []
        for section_name in FULL_ONLY_SECTION_ORDER:
            section = payload.get(section_name)
            if section is None:
                missing_aggregate_fields.append(section_name)
                continue
            for field in AGG_REQUIRED_FIELDS[section_name]:
                if field not in section:
                    missing_aggregate_fields.append(f"{section_name}.{field}")
        if missing_aggregate_fields:
            return "missing_aggregate_fields", sorted(missing_aggregate_fields)

    return None, []


def determine_coverage_tier(payload: dict) -> str:
    """Classify payloads for the REDUCED/FULL API contract."""

    if not isinstance(payload, dict):
        raise ValueError("payload must be a dict")
    if "application" not in payload or payload.get("application") is None:
        raise ValueError("application is required")

    top_keys = set(payload)
    if top_keys == {"application"}:
        return "REDUCED"
    if top_keys == set(FULL_SECTION_ORDER):
        return "FULL"
    if top_keys.intersection(FULL_ONLY_SECTIONS):
        raise ValueError("partial full payload")
    raise ValueError("unsupported starter payload")


def build_input_df(payload: dict, tier: str) -> pd.DataFrame:
    """Flatten a valid request payload into a one-row DataFrame."""

    tier = tier.upper()
    if tier not in {"FULL", "REDUCED"}:
        raise ValueError("tier must be FULL or REDUCED")

    flattened: dict[str, Any] = {}
    section_names = ("application",) if tier == "REDUCED" else FULL_SECTION_ORDER

    for section_name in section_names:
        section = payload.get(section_name)
        if section_name == "application" and section is None:
            raise _api_error("missing_application", ["application"])
        if not isinstance(section, dict):
            raise _api_error("bad_request")

        for field_name, field_value in section.items():
            if not _is_scalar_value(field_value):
                raise _api_error("bad_request")
            if field_name in flattened:
                raise _api_error("bad_request")
            flattened[field_name] = field_value

    return pd.DataFrame([flattened])


def score_request(payload: dict, runtime: ApiRuntime, mock_mode: bool = False) -> dict:
    """Score one request using the loaded immutable runtime."""

    error_code, missing_fields = validate_payload(payload)
    if error_code is not None:
        raise _api_error(error_code, missing_fields)

    tier = determine_coverage_tier(payload)
    input_df = build_input_df(payload, tier)
    builder, model, calibrator, explainer = _get_tier_runtime_components(runtime, tier)

    features = _invoke_builder(builder, input_df)
    raw_pd = _predict_raw_pd(model, features)
    calibrated_pd = _calibrate_pd(calibrator, raw_pd)
    decision = _decision_from_pd(calibrated_pd)
    explanations = (
        _mock_top_5_explanations(features.columns)
        if mock_mode
        else _compute_real_top_5_explanations(explainer, features)
    )

    return {
        "probability_of_default": calibrated_pd,
        "decision": decision,
        "escalate": decision == "REVIEW",
        "top_5_explanations": explanations,
        "model_version": runtime.tier_model_versions[tier],
        "calibrated": True,
        "model_fairness_audit_passed": runtime.model_fairness_audit_passed,
        "fairness_audit_version": FAIRNESS_AUDIT_VERSION,
        "coverage_tier": tier,
    }


def _resolve_runtime_dirs(
    artifact_dir: str | None,
    processed_dir: str | None,
    mock_mode: bool,
) -> tuple[str, str]:
    if mock_mode:
        resolved_artifact_dir = artifact_dir or tempfile.mkdtemp(prefix="mastermind_mock_artifacts_")
        resolved_processed_dir = processed_dir or tempfile.mkdtemp(prefix="mastermind_mock_processed_")
    else:
        resolved_artifact_dir = artifact_dir or ARTIFACT_DIR
        resolved_processed_dir = processed_dir or DATA_DIR
    return (os.path.abspath(resolved_artifact_dir), os.path.abspath(resolved_processed_dir))


def _build_mock_runtime(artifact_dir: str, processed_dir: str) -> ApiRuntime:
    full_builder = _MockBuilder("FULL")
    full_model = _MockModel(len(full_builder.encoded_columns_))
    full_calibrator = _MockCalibrator()
    full_explainer = _MockExplainer()
    reduced_builder = _MockBuilder("REDUCED")
    reduced_model = _MockModel(len(reduced_builder.encoded_columns_))
    reduced_calibrator = _MockCalibrator()
    reduced_explainer = _MockExplainer()

    processed_manifest = MappingProxyType(
        {
            "mode": "mock",
            "processed_manifest_id": "mock-processed-manifest",
        }
    )
    health_model_version = _build_composite_model_version("FULL")
    tier_model_versions = MappingProxyType(
        {
            "FULL": _build_composite_model_version("FULL"),
            "REDUCED": _build_composite_model_version("REDUCED"),
        }
    )

    return ApiRuntime(
        artifact_dir=artifact_dir,
        processed_dir=processed_dir,
        processed_manifest=processed_manifest,
        full_builder=full_builder,
        full_model=full_model,
        full_calibrator=full_calibrator,
        full_shap_explainer=full_explainer,
        reduced_builder=reduced_builder,
        reduced_model=reduced_model,
        reduced_calibrator=reduced_calibrator,
        reduced_shap_explainer=reduced_explainer,
        model_fairness_audit_passed=False,
        health_model_version=health_model_version,
        tier_model_versions=tier_model_versions,
        coverage_tiers_available=("FULL", "REDUCED"),
        reproducibility_report=MappingProxyType({"mode": "mock"}),
        mock_mode=True,
    )


def _load_real_runtime(
    artifact_dir: str,
    processed_dir: str,
    strict_artifacts: bool,
) -> ApiRuntime:
    processed_manifest = _load_processed_manifest(processed_dir)
    builder_module = _import_builder_artifacts_module()
    builders = _load_validated_builders(
        builder_module,
        artifact_dir=artifact_dir,
        processed_dir=processed_dir,
        processed_manifest=processed_manifest,
        strict_artifacts=strict_artifacts,
    )
    full_builder = builders["FULL"]
    reduced_builder = builders["REDUCED"]

    full_model = _load_joblib_artifact(artifact_dir, "full_model.joblib", "FULL model")
    full_calibrator = _load_joblib_artifact(artifact_dir, "full_calibrator.joblib", "FULL calibrator")
    full_explainer = _load_joblib_artifact(
        artifact_dir, "full_shap_explainer.joblib", "FULL SHAP explainer"
    )
    reduced_model = _load_joblib_artifact(artifact_dir, "reduced_model.joblib", "REDUCED model")
    reduced_calibrator = _load_joblib_artifact(
        artifact_dir,
        "reduced_calibrator.joblib",
        "REDUCED calibrator",
    )
    reduced_explainer = _load_joblib_artifact(
        artifact_dir,
        "reduced_shap_explainer.joblib",
        "REDUCED SHAP explainer",
    )
    fairness_result = _load_fairness_result(artifact_dir)
    reproducibility_report = _load_optional_json(artifact_dir, REPRODUCIBILITY_REPORT_FILENAME)

    _validate_tier_runtime("FULL", full_builder, full_model, full_calibrator, full_explainer)
    _validate_tier_runtime(
        "REDUCED",
        reduced_builder,
        reduced_model,
        reduced_calibrator,
        reduced_explainer,
    )
    health_model_version = _resolve_health_model_version(
        reproducibility_report,
        fallback=_build_composite_model_version("FULL"),
    )
    tier_model_versions = MappingProxyType(
        {
            "FULL": _resolve_tier_model_version(reproducibility_report, "FULL"),
            "REDUCED": _resolve_tier_model_version(reproducibility_report, "REDUCED"),
        }
    )

    return ApiRuntime(
        artifact_dir=artifact_dir,
        processed_dir=processed_dir,
        processed_manifest=MappingProxyType(processed_manifest),
        full_builder=full_builder,
        full_model=full_model,
        full_calibrator=full_calibrator,
        full_shap_explainer=full_explainer,
        reduced_builder=reduced_builder,
        reduced_model=reduced_model,
        reduced_calibrator=reduced_calibrator,
        reduced_shap_explainer=reduced_explainer,
        model_fairness_audit_passed=fairness_result,
        health_model_version=health_model_version,
        tier_model_versions=tier_model_versions,
        coverage_tiers_available=("FULL", "REDUCED"),
        reproducibility_report=MappingProxyType(reproducibility_report),
        mock_mode=False,
    )


def _load_processed_manifest(processed_dir: str) -> dict[str, Any]:
    manifest_path = os.path.join(processed_dir, PROCESSED_MANIFEST_FILENAME)
    if not os.path.exists(manifest_path):
        raise RuntimeError(f"Missing required processed manifest: {manifest_path}")
    try:
        with open(manifest_path, "r", encoding="utf-8") as handle:
            manifest = json.load(handle)
    except Exception as exc:
        raise RuntimeError(f"Failed to read processed manifest: {manifest_path}") from exc
    if not isinstance(manifest, dict):
        raise RuntimeError("processed_artifact_manifest.json must contain a JSON object")
    return manifest


def _import_builder_artifacts_module() -> ModuleType:
    try:
        return importlib.import_module("src.builder_artifacts")
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "src.builder_artifacts.py is required for strict builder loading"
        ) from exc


def _load_validated_builders(
    builder_module: ModuleType,
    *,
    artifact_dir: str,
    processed_dir: str,
    processed_manifest: Mapping[str, Any],
    strict_artifacts: bool,
) -> Any:
    loader = getattr(builder_module, "load_validated_builders", None)
    if not callable(loader):
        raise RuntimeError(
            "src.builder_artifacts.py must expose load_validated_builders(...)"
        )
    result = _call_with_supported_kwargs(
        loader,
        artifact_dir=artifact_dir,
        processed_dir=processed_dir,
        processed_manifest=processed_manifest,
        strict_artifacts=strict_artifacts,
    )
    return _coerce_tier_builders(result)


def _coerce_tier_builders(result: Any) -> dict[str, Any]:
    if isinstance(result, dict):
        full_builder = result.get("full_builder") or result.get("FULL") or result.get("full")
        reduced_builder = (
            result.get("reduced_builder") or result.get("REDUCED") or result.get("reduced")
        )
        if full_builder is not None and reduced_builder is not None:
            return {"FULL": full_builder, "REDUCED": reduced_builder}

    if isinstance(result, (tuple, list)) and len(result) >= 2:
        return {"FULL": result[0], "REDUCED": result[1]}

    full_builder = getattr(result, "full_builder", None)
    reduced_builder = getattr(result, "reduced_builder", None)
    if full_builder is not None and reduced_builder is not None:
        return {"FULL": full_builder, "REDUCED": reduced_builder}

    raise RuntimeError("Builder loader must return both FULL and REDUCED builders")


def _call_with_supported_kwargs(func: Any, **kwargs: Any) -> Any:
    signature = inspect.signature(func)
    parameters = signature.parameters
    if any(param.kind == inspect.Parameter.VAR_KEYWORD for param in parameters.values()):
        return func(**kwargs)
    supported_kwargs = {key: value for key, value in kwargs.items() if key in parameters}
    return func(**supported_kwargs)


def _load_joblib_artifact(artifact_dir: str, filename: str, label: str) -> Any:
    path = os.path.join(artifact_dir, filename)
    if not os.path.exists(path):
        raise RuntimeError(f"Missing required {label}: {path}")
    try:
        return joblib.load(path)
    except Exception as exc:
        raise RuntimeError(f"Failed to load {label}: {path}") from exc


def _load_optional_json(artifact_dir: str, filename: str) -> dict[str, Any]:
    path = os.path.join(artifact_dir, filename)
    if not os.path.exists(path):
        return {}
    try:
        with open(path, "r", encoding="utf-8") as handle:
            data = json.load(handle)
    except Exception as exc:
        raise RuntimeError(f"Failed to read JSON metadata: {path}") from exc
    if not isinstance(data, dict):
        raise RuntimeError(f"JSON metadata must be an object: {path}")
    return data


def _load_fairness_result(artifact_dir: str) -> bool:
    fairness_obj = _load_joblib_artifact(
        artifact_dir,
        FAIRNESS_RESULT_FILENAME,
        "fairness result artifact",
    )
    if isinstance(fairness_obj, (bool, np.bool_)):
        return bool(fairness_obj)
    raise RuntimeError(
        "model_fairness_audit_passed.joblib must contain a boolean result"
    )


def _validate_tier_runtime(
    tier: str,
    builder: Any,
    model: Any,
    calibrator: Any,
    explainer: Any,
) -> None:
    smoke_payload = _build_smoke_payload(tier)
    smoke_df = build_input_df(smoke_payload, tier)
    features = _invoke_builder(builder, smoke_df)
    validate_transformed_frame(features, expected_rows=1)

    n_features = getattr(model, "n_features_in_", None)
    if n_features is not None and int(n_features) != features.shape[1]:
        raise RuntimeError(
            f"{tier} model expects {int(n_features)} features but builder produced {features.shape[1]}"
        )

    raw_pd = _predict_raw_pd(model, features)
    calibrated_pd = _calibrate_pd(calibrator, raw_pd)
    if not 0.0 <= calibrated_pd <= 1.0:
        raise RuntimeError(f"{tier} calibrator produced an invalid probability")

    _compute_real_top_5_explanations(explainer, features)


def _build_smoke_payload(tier: str) -> dict[str, Any]:
    sample_payload = _build_demo_seed_payload()
    if tier.upper() == "REDUCED":
        return {"application": sample_payload["application"]}
    return sample_payload


def _build_demo_seed_payload() -> dict[str, Any]:
    application = {
        "AMT_INCOME_TOTAL_CAPPED": 120000.0,
        "AMT_CREDIT": 250000.0,
        "AMT_ANNUITY": 25000.0,
        "AMT_GOODS_PRICE": 220000.0,
        "DAYS_BIRTH": -12000.0,
        "DAYS_EMPLOYED": -1500.0,
        "DAYS_REGISTRATION": -3000.0,
        "DAYS_ID_PUBLISH": -2000.0,
        "DAYS_LAST_PHONE_CHANGE": -1000.0,
        "REGION_POPULATION_RELATIVE": 0.02,
        "EXT_SOURCE_1": 0.2,
        "EXT_SOURCE_2": 0.4,
        "EXT_SOURCE_3": 0.6,
        "CNT_FAM_MEMBERS": 2.0,
        "OWN_CAR_AGE": 5.0,
        "OBS_30_CNT_SOCIAL_CIRCLE": 1.0,
        "DEF_30_CNT_SOCIAL_CIRCLE": 0.0,
        "OBS_60_CNT_SOCIAL_CIRCLE": 1.0,
        "DEF_60_CNT_SOCIAL_CIRCLE": 0.0,
        "AMT_REQ_CREDIT_BUREAU_HOUR": 0.0,
        "AMT_REQ_CREDIT_BUREAU_DAY": 0.0,
        "AMT_REQ_CREDIT_BUREAU_WEEK": 1.0,
        "AMT_REQ_CREDIT_BUREAU_MON": 1.0,
        "AMT_REQ_CREDIT_BUREAU_QRT": 0.0,
        "AMT_REQ_CREDIT_BUREAU_YEAR": 1.0,
        "NAME_CONTRACT_TYPE": "Cash loans",
        "NAME_TYPE_SUITE": "Unaccompanied",
        "NAME_EDUCATION_TYPE": "Higher education",
        "NAME_FAMILY_STATUS": "Married",
        "OCCUPATION_TYPE": "Laborers",
        "ORGANIZATION_TYPE": "Business Entity Type 3",
        "WEEKDAY_APPR_PROCESS_START": "MONDAY",
        "DAYS_EMPLOYED_ANOM": 0,
    }
    return {
        "application": application,
        "bureau_agg": {
            "BUREAU_LOAN_COUNT": 2.0,
            "BUREAU_ACTIVE_COUNT": 1.0,
            "BUREAU_CLOSED_COUNT": 1.0,
            "BUREAU_AMT_CREDIT_SUM_SUM": 50000.0,
            "BUREAU_AMT_CREDIT_SUM_DEBT_SUM": 10000.0,
            "BUREAU_DEBT_TO_CREDIT_RATIO": 0.2,
            "BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM": 0.0,
            "BUREAU_CREDIT_DAY_OVERDUE_MAX": 0.0,
            "BUREAU_DAYS_CREDIT_MAX": -200.0,
            "BUREAU_CNT_CREDIT_PROLONG_SUM": 0.0,
        },
        "previous_agg": {
            "PREV_APP_COUNT": 3.0,
            "PREV_APPROVED_COUNT": 2.0,
            "PREV_REFUSED_COUNT": 1.0,
            "PREV_APPROVAL_RATE": 0.67,
            "PREV_REFUSAL_RATE": 0.33,
            "PREV_AMT_APPLICATION_MEAN": 210000.0,
            "PREV_AMT_CREDIT_MEAN": 195000.0,
            "PREV_AMT_GOODS_PRICE_MEAN": 187000.0,
            "PREV_APP_CREDIT_DIFF_MEAN": 15000.0,
            "PREV_DAYS_DECISION_MAX": -120.0,
            "PREV_RATE_DOWN_PAYMENT_MEAN": 0.08,
        },
        "installments_agg": {
            "INST_RECORD_COUNT": 12.0,
            "INST_MISSED_RATE": 0.08,
            "INST_DPD_MEAN": 4.0,
            "INST_DPD_MAX": 12.0,
            "INST_PAYMENT_RATIO_MEAN": 0.95,
            "INST_PAYMENT_RATIO_MIN": 0.72,
            "INST_LATE_COUNT": 3.0,
        },
        "pos_cash_agg": {
            "POS_RECORD_COUNT": 6.0,
            "POS_DPD_MEAN": 1.5,
            "POS_DPD_MAX": 7.0,
            "POS_DPD_DEF_MEAN": 0.5,
            "POS_DPD_DEF_MAX": 4.0,
            "POS_COMPLETED_RATE": 0.6,
            "POS_ACTIVE_RATE": 0.4,
            "POS_CNT_INSTALMENT_FUTURE_MEAN": 2.0,
        },
        "credit_card_agg": {
            "CC_RECORD_COUNT": 8.0,
            "CC_BALANCE_MEAN": 18000.0,
            "CC_LIMIT_MEAN": 60000.0,
            "CC_UTILIZATION_MEAN": 0.3,
            "CC_PAYMENT_RATIO_MEAN": 1.1,
            "CC_DPD_MEAN": 1.0,
            "CC_DPD_MAX": 6.0,
            "CC_DRAWINGS_ATM_SUM": 4500.0,
            "CC_DRAWINGS_CURRENT_SUM": 9000.0,
        },
    }


def _build_demo_config(runtime: ApiRuntime) -> dict[str, Any]:
    sample_payload = _build_demo_seed_payload()
    sections: list[dict[str, Any]] = []

    for section_name in FULL_SECTION_ORDER:
        field_names = (
            APPLICATION_REQUIRED_FIELDS
            if section_name == "application"
            else AGG_REQUIRED_FIELDS[section_name]
        )
        fields = []
        for field_name in field_names:
            field_kind = "select" if field_name in CATEGORICAL_APPLICATION_FIELDS else "number"
            fields.append(
                {
                    "name": field_name,
                    "kind": field_kind,
                    "options": list(DEMO_FIELD_OPTIONS.get(field_name, ())),
                }
            )
        sections.append(
            {
                "name": section_name,
                "label": section_name.replace("_", " ").title(),
                "fields": fields,
            }
        )

    return {
        "runtimeMode": "mock" if runtime.mock_mode else "real",
        "healthModelVersion": runtime.health_model_version,
        "scoreModelVersions": dict(runtime.tier_model_versions),
        "fairnessAuditPassed": runtime.model_fairness_audit_passed,
        "fairnessAuditVersion": FAIRNESS_AUDIT_VERSION,
        "defaultTier": UI_DEFAULT_TIER,
        "sections": sections,
        "samplePayloads": {
            "FULL": sample_payload,
            "REDUCED": {"application": sample_payload["application"]},
        },
        "routes": {
            "health": "/health",
            "score": "/score",
            "home": "/",
            "analyze": "/analyze",
            "status": "/status",
        },
    }


def _build_health_snapshot(runtime: ApiRuntime) -> dict[str, Any]:
    return {
        "status": "ok",
        "model_version": runtime.health_model_version,
        "fairness_audit_passed": runtime.model_fairness_audit_passed,
        "coverage_tiers_available": list(runtime.coverage_tiers_available),
    }


def _get_tier_runtime_components(runtime: ApiRuntime, tier: str) -> tuple[Any, Any, Any, Any]:
    tier = tier.upper()
    if tier == "FULL":
        return (
            runtime.full_builder,
            runtime.full_model,
            runtime.full_calibrator,
            runtime.full_shap_explainer,
        )
    if tier == "REDUCED":
        return (
            runtime.reduced_builder,
            runtime.reduced_model,
            runtime.reduced_calibrator,
            runtime.reduced_shap_explainer,
        )
    raise RuntimeError(f"Unsupported tier: {tier}")


def _invoke_builder(builder: Any, df: pd.DataFrame) -> pd.DataFrame:
    if hasattr(builder, "transform") and callable(builder.transform):
        result = builder.transform(df)
    elif callable(builder):
        result = builder(df)
    else:
        raise RuntimeError("Loaded builder is neither callable nor transformable")

    if not isinstance(result, pd.DataFrame):
        raise RuntimeError("Builder output must be a pandas DataFrame")
    if result.shape[0] != df.shape[0]:
        raise RuntimeError("Builder output row count does not match input row count")
    return result


def _predict_raw_pd(model: Any, features: pd.DataFrame) -> float:
    if not hasattr(model, "predict_proba") or not callable(model.predict_proba):
        raise RuntimeError("Loaded model does not expose predict_proba")
    probs = np.asarray(model.predict_proba(features), dtype=float)
    if probs.ndim != 2 or probs.shape[0] != 1 or probs.shape[1] < 2:
        raise RuntimeError("Model predict_proba must return shape (1, >=2)")
    return float(probs[0, 1])


def _calibrate_pd(calibrator: Any, raw_pd: float) -> float:
    if not hasattr(calibrator, "predict") or not callable(calibrator.predict):
        raise RuntimeError("Loaded calibrator does not expose predict")
    calibrated = np.asarray(calibrator.predict(np.array([raw_pd], dtype=float)), dtype=float).reshape(-1)
    if calibrated.size != 1:
        raise RuntimeError("Calibrator predict must return exactly one probability")
    return float(np.clip(calibrated[0], 0.0, 1.0))


def _decision_from_pd(probability_of_default: float) -> str:
    if probability_of_default < APPROVE_THRESHOLD:
        return "APPROVE"
    if probability_of_default < DECLINE_THRESHOLD:
        return "REVIEW"
    return "DECLINE"


def _compute_real_top_5_explanations(explainer: Any, features: pd.DataFrame) -> list[dict[str, str]]:
    shap_series = _compute_shap_series(explainer, features)
    explanations = top_5_explanations_from_shap(shap_series)
    return _ensure_five_explanations(explanations, list(shap_series.index))


def _compute_shap_series(explainer: Any, features: pd.DataFrame) -> pd.Series:
    raw_input = features.to_numpy(dtype=float, copy=False)
    if callable(explainer):
        raw_shap = explainer(raw_input)
    elif hasattr(explainer, "shap_values") and callable(explainer.shap_values):
        raw_shap = explainer.shap_values(raw_input)
    else:
        raise RuntimeError("Loaded explainer is not callable and has no shap_values method")
    values = _normalize_shap_output(raw_shap, len(features.columns))
    return pd.Series(values, index=features.columns, dtype=float)


def _normalize_shap_output(raw_shap: Any, feature_count: int) -> np.ndarray:
    try:
        import shap  # type: ignore
    except Exception:  # pragma: no cover - dependency import differences are environment-specific
        shap = None

    if shap is not None and isinstance(raw_shap, shap.Explanation):
        raw_shap = raw_shap.values

    if isinstance(raw_shap, list):
        if not raw_shap:
            raise RuntimeError("Explainer returned an empty SHAP list")
        raw_shap = raw_shap[1] if len(raw_shap) > 1 else raw_shap[0]

    values = np.asarray(raw_shap, dtype=float)
    if values.ndim == 3:
        class_index = 1 if values.shape[-1] > 1 else 0
        values = values[..., class_index]
    if values.ndim == 2:
        if values.shape[0] == 1:
            values = values[0]
        elif values.shape[1] == 1:
            values = values[:, 0]
    if values.ndim != 1 or values.shape[0] != feature_count:
        raise RuntimeError("Explainer returned SHAP values with an unexpected shape")
    return values


def _ensure_five_explanations(
    explanations: list[dict[str, str]],
    feature_names: list[str],
) -> list[dict[str, str]]:
    if len(explanations) >= 5:
        return explanations[:5]
    used = {item["feature"] for item in explanations}
    for feature_name in feature_names:
        if feature_name in used:
            continue
        explanations.append({"feature": feature_name, "reason": render_reason(feature_name)})
        used.add(feature_name)
        if len(explanations) == 5:
            break
    while len(explanations) < 5:
        fallback_feature = f"fallback_feature_{len(explanations) + 1}"
        explanations.append({"feature": fallback_feature, "reason": render_reason(fallback_feature)})
    return explanations


def _mock_top_5_explanations(feature_names: Any) -> list[dict[str, str]]:
    ordered_names = list(feature_names)[:5]
    if len(ordered_names) < 5:
        ordered_names.extend(f"mock_feature_{index}" for index in range(len(ordered_names), 5))
    return [{"feature": name, "reason": render_reason(name)} for name in ordered_names[:5]]


def _coerce_2d_numeric(value: Any) -> np.ndarray:
    if isinstance(value, pd.DataFrame):
        arr = value.to_numpy(dtype=float, copy=False)
    else:
        arr = np.asarray(value, dtype=float)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    return arr


def _numeric_series(df: pd.DataFrame, column_name: str, default: float = 0.0) -> pd.Series:
    if column_name in df.columns:
        return pd.to_numeric(df[column_name], errors="coerce").fillna(default)
    return pd.Series(default, index=df.index, dtype=float)


def _section_has_scalar_values(section: Mapping[str, Any]) -> bool:
    return all(_is_scalar_value(value) for value in section.values())


def _is_scalar_value(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, (str, int, float, bool, np.generic)):
        return True
    return False


def _build_composite_model_version(tier: str) -> str:
    tier_key = tier.lower()
    tier_version = MODEL_VERSIONS.get(tier_key, f"{tier_key}_unknown")
    fairness_tag = _fairness_tag_from_version(FAIRNESS_AUDIT_VERSION)
    return f"{tier_version}|{ROUTER_VERSION}|{POLICY_VERSION}|fairness_v{fairness_tag}"


def _fairness_tag_from_version(version: str) -> str:
    match = re.search(r"(20\d{2}Q[1-4])", version)
    return match.group(1) if match else DEFAULT_FAIRNESS_VERSION_TAG


def _resolve_health_model_version(
    reproducibility_report: Mapping[str, Any],
    *,
    fallback: str,
) -> str:
    preferred_keys = (
        "deployed_model_version",
        "health_model_version",
        "model_version",
        "champion_model_version",
        "full_model_version",
    )
    discovered = _find_first_string_value(reproducibility_report, preferred_keys)
    return discovered or fallback


def _resolve_tier_model_version(
    reproducibility_report: Mapping[str, Any],
    tier: str,
) -> str:
    tier_key = tier.lower()
    preferred_keys = (
        f"{tier_key}_model_version",
        f"{tier_key}_deployed_model_version",
        f"{tier_key}_version",
    )
    discovered = _find_first_string_value(reproducibility_report, preferred_keys)
    return discovered or _build_composite_model_version(tier)


def _find_first_string_value(data: Any, keys: tuple[str, ...]) -> str | None:
    if isinstance(data, dict):
        for key in keys:
            value = data.get(key)
            if isinstance(value, str) and value:
                return value
        for value in data.values():
            found = _find_first_string_value(value, keys)
            if found:
                return found
    elif isinstance(data, list):
        for value in data:
            found = _find_first_string_value(value, keys)
            if found:
                return found
    return None


def _get_runtime(app: Flask) -> ApiRuntime:
    runtime = app.extensions.get(RUNTIME_EXTENSION_KEY)
    if runtime is None:
        raise RuntimeError("API runtime is not initialized")
    return runtime


def _api_error(error_code: str, missing_fields: list[str] | tuple[str, ...] | None = None) -> ApiError:
    status_code = 400 if error_code == "bad_request" else 422
    return ApiError(
        status_code=status_code,
        error_code=error_code,
        message=ERROR_MESSAGES[error_code],
        missing_fields=tuple(missing_fields or ()),
    )


__all__ = [
    "APPLICATION_REQUIRED_FIELDS",
    "AGG_REQUIRED_FIELDS",
    "ApiRuntime",
    "build_input_df",
    "create_app",
    "determine_coverage_tier",
    "score_request",
    "validate_payload",
]


## Module 5: App Entrypoint\n
\n
This small entrypoint resolves mock mode and launches the Flask app.\n
\n
Original source: `app.py`\n

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from src.api.app import create_app


PROJECT_ROOT = Path(__file__).resolve().parent
_TRUE_VALUES = {"1", "true", "yes", "on"}
_FALSE_VALUES = {"0", "false", "no", "off"}

_REAL_RUNTIME_REQUIRED_PATHS = (
    PROJECT_ROOT / "artifacts" / "full_feature_builder.joblib",
    PROJECT_ROOT / "artifacts" / "full_feature_builder.manifest.json",
    PROJECT_ROOT / "artifacts" / "full_model.joblib",
    PROJECT_ROOT / "artifacts" / "full_calibrator.joblib",
    PROJECT_ROOT / "artifacts" / "full_shap_explainer.joblib",
    PROJECT_ROOT / "artifacts" / "reduced_feature_builder.joblib",
    PROJECT_ROOT / "artifacts" / "reduced_feature_builder.manifest.json",
    PROJECT_ROOT / "artifacts" / "reduced_model.joblib",
    PROJECT_ROOT / "artifacts" / "reduced_calibrator.joblib",
    PROJECT_ROOT / "artifacts" / "reduced_shap_explainer.joblib",
    PROJECT_ROOT / "artifacts" / "model_fairness_audit_passed.joblib",
    PROJECT_ROOT / "data" / "processed" / "processed_artifact_manifest.json",
)


def _env_flag(name: str) -> bool | None:
    value = os.getenv(name, "").strip().lower()
    if not value:
        return None
    if value in _TRUE_VALUES:
        return True
    if value in _FALSE_VALUES:
        return False
    return None


def _has_complete_real_runtime() -> bool:
    return all(path.exists() for path in _REAL_RUNTIME_REQUIRED_PATHS)


def _resolve_mock_mode() -> bool:
    explicit = _env_flag("MASTERMIND_MOCK_MODE")
    if explicit is not None:
        return explicit
    return not _has_complete_real_runtime()


app = create_app(mock_mode=_resolve_mock_mode())


if __name__ == "__main__":
    host = os.getenv("FLASK_RUN_HOST", "127.0.0.1")
    port = int(os.getenv("FLASK_RUN_PORT", "5000"))
    app.run(host=host, port=port)
